# CEDAR Downstream Supervised Training (Cell C: full unfreeze) - Kaggle Notebook

Step 4 Cell C of the CEDAR **in-domain** rung
(`docs/claude_response/downstream_supervised_learning_approach.md`, §18/§20):
takes the CEDAR-only DenseCL-pretrained encoder (`CEDAR_data_ssl/fold_0`,
epoch 50 - the smallest of the three single-dataset SSL pools, 35 train /
5 val writers) and fine-tunes it for signature verification with the
**entire encoder unfrozen** (stem + stage1-4) - the last, highest-capacity
rung of the frozen -> stage4 -> full-unfreeze ladder, mirroring the
already-completed Hindi and Bengali in-domain Cell C runs exactly, just on
CEDAR's own SSL encoder and CEDAR's fold_0 writers (35 train / 5 val / 15
test).

Cell A (frozen) and Cell B (stage4) for CEDAR are running locally on two
separate machines in parallel with this notebook - only Cell C is on
Kaggle here, since it is the longest/highest-capacity rung and a GPU slot
was free.

**How this notebook is organized** (same convention as the Bengali Cell
B/C notebooks / the project's existing DenseCL SSL pretraining Kaggle
notebook):
1. One single editable cell (`kaggle_config.py`) holds every input path,
   the margins, and every training hyperparameter - edit that cell only.
2. A few setup cells install/check dependencies and confirm the GPU.
3. A cell that writes the exact, already-frozen fold_0 CEDAR test/
   validation writer splits (identical to every local CEDAR run) - copied
   in verbatim, not regenerated.
4. One `%%writefile` cell per source module - the exact same logic as the
   local project's `supervised_verification_approach/` code, just
   flattened into one directory (no `sys.path` bootstrapping needed) and
   re-pointed at `kaggle_config.py` for paths.
5. The training run itself, then a small results-verification cell.

**Before running - TWO separate Kaggle Dataset inputs needed:**
1. **Signature images**: `CEDAR/<writer_id>/<file>`.
2. **The CEDAR-only SSL encoder checkpoint** at the relative path
   `CEDAR_data_ssl/fold_0/checkpoints/encoder_epoch50.pt` inside the
   dataset - either add it to the same checkpoint dataset already used for
   the Bengali Cell B/C runs (alongside `Bengali_data_ssl/`), or attach a
   separate dataset that contains it, and adjust `SSL_RESULTS_ROOT` in the
   config cell to point at whichever folder directly contains
   `CEDAR_data_ssl/`.

`DATA_ROOT` / `SSL_RESULTS_ROOT` in the config cell are pre-filled with
the values already confirmed working for the Bengali Cell B/C Kaggle
runs. If your dataset slugs/mount paths differ, run `!ls /kaggle/input/`
and fix them, then **Commit and Run All**.


## 1. Setup

In [ ]:
# Kaggle's base image already ships torch + CUDA and (usually) opencv/scikit-learn,
# but this makes the notebook self-contained even on a fresh/changed image.
!pip install -q opencv-python-headless pandas tqdm scikit-learn


In [ ]:
import torch

print(f"Torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

!nvidia-smi


## 2. Configuration - the ONLY cell you should need to edit

Run the next cell first (`!ls /kaggle/input/`) to see your attached datasets'
actual mounted folder names, THEN edit `DATA_ROOT` / `SSL_RESULTS_ROOT` in
the `kaggle_config.py` cell below to match before running the rest.

In [ ]:
# Run this BEFORE editing kaggle_config.py below, to find the exact mounted
# folder names for your two attached datasets (images + SSL checkpoint).
!ls /kaggle/input/


In [ ]:
import os

os.makedirs("/kaggle/working/supervised_src", exist_ok=True)


In [ ]:
%%writefile /kaggle/working/supervised_src/kaggle_config.py
from __future__ import annotations

from pathlib import Path

# ============================================================================
# EDIT THIS SECTION - every input path and training knob lives here.
# ============================================================================
#
# This notebook runs Step 4 Cell C of the CEDAR IN-DOMAIN rung
# (downstream_supervised_learning_approach.md SS18/SS20): the CEDAR-only
# DenseCL-pretrained encoder (CEDAR_data_ssl/fold_0, 35 train / 5 val
# writers - the smallest of the three single-dataset SSL pools), with the
# FULL encoder unfrozen (stem + stage1-4) - the last/most-capacity rung of
# the frozen -> stage4 -> full-unfreeze ladder. Mirrors the completed
# Hindi/Bengali in-domain Cell C runs exactly, just on CEDAR's own SSL
# encoder and CEDAR's fold_0 writers (35 train / 5 val / 15 test). Cell A
# (frozen) and Cell B (stage4) for CEDAR are running locally on two
# separate machines in parallel with this Cell C Kaggle run - only this
# notebook's Cell C is on Kaggle, since it's the highest-capacity/longest
# rung and a free GPU slot was available.
#
# Two SEPARATE Kaggle Dataset inputs are needed (same convention as the
# Bengali Cell B/C notebooks):
#   1. The signature image dataset (CEDAR/<writer_id>/<file>).
#   2. The CEDAR-only SSL encoder checkpoint, at the relative path
#      "CEDAR_data_ssl/fold_0/checkpoints/encoder_epoch50.pt".
#
# DATA_ROOT / SSL_RESULTS_ROOT below are pre-filled with the values
# CONFIRMED WORKING for the Bengali Cell B/C Kaggle runs (same account,
# same dataset-mount convention) - if you attach the SAME two Kaggle
# Datasets (the CEDAR_data_ssl SSL-checkpoint dataset must also contain
# CEDAR_data_ssl/fold_0/checkpoints/encoder_epoch50.pt, alongside or
# instead of the Bengali one), no edit should be needed; otherwise run
# `!ls /kaggle/input/` and fix them to match.

# --- Paths -------------------------------------------------------------------
# Must point at the folder that DIRECTLY contains CEDAR/ (only this one
# dataset folder is needed for this run - not all three).
DATA_ROOT = Path("/kaggle/input/datasets/lihazinaveed/signature-dataset-all/all/all")

# Must point at the folder that DIRECTLY contains CEDAR_data_ssl/ (i.e.
# one level ABOVE the "CEDAR_data_ssl/fold_0/checkpoints/encoder_epoch50.pt"
# path inside your uploaded checkpoint dataset).
SSL_RESULTS_ROOT = Path("/kaggle/input/datasets/naveedlihazi/signature-verification-dense-cl-approach")

# Everything this run produces (training_history.csv, checkpoints) lands
# here, mirroring the local project's results/<run_name>/<dataset>/<run_tag>/
# layout exactly, so it drops straight into the same place if copied back
# into the local repo after downloading Kaggle's Output. /kaggle/working/ is
# what "Commit and Run All" persists as the notebook's output.
RESULTS_DIR = Path("/kaggle/working/results") / "CEDAR_data_ssl/fold_0" / "CEDAR" / "step4_full_unfrozen_combined_indomain"

# --- Run identity --------------------------------------------------------------
FOLD = "fold_0"
DATASET_NAME = "CEDAR"
RUN_NAME = "CEDAR_data_ssl/fold_0"          # which SSL encoder to load (and where results are keyed)
CHECKPOINT_EPOCH = 50                        # pinned: the completed 50-epoch CEDAR-only DenseCL run
RUN_TAG = "step4_full_unfrozen_combined_indomain"  # Cell C: full unfreeze - matches Hindi/Bengali's on-disk naming

# -- Model / encoder unfreezing --------------------------------------------------
# Cell C: the WHOLE encoder (stem + every stage) gets gradients - the
# "ultimate capacity check" rung. Everything else (margins, batch size,
# learning rates) is unchanged from the local Cell A/B runs on this same
# CEDAR-only encoder - only this one setting differs between cells.
TRAINABLE_ENCODER_STAGES: tuple[str, ...] = ("stem", "stage1", "stage2", "stage3", "stage4")
PROJECTOR_HIDDEN_DIM = 256
EMBEDDING_DIM = 256
NORM_TYPE = "batch"
LOCAL_EMBEDDING_DIM = 128   # Step 4's local/structural branch (DetailSemNet Eq. 3/4)
LAMBDA_0 = 1.0              # dis = LAMBDA_0 * dis_global + dis_struct

# -- Loss: double-margin combined-distance ---------------------------------------
# THE CEDAR IN-DOMAIN MARGINS (2026-09-08) - swept specifically against
# THIS SSL encoder (CEDAR_data_ssl/fold_0), NOT the pooled-encoder CEDAR
# margins (0.33/0.67) - margins are a property of the specific encoder's
# raw embedding-space distance scale, not portable across different SSL
# runs, and NOT of how many encoder stages happen to be unfrozen
# downstream (the sweep proxy always uses the raw SSL checkpoint
# regardless). Swept via sweep_margins_combined.py --dataset CEDAR
# --skip_step3_encoder --ssl_run_name CEDAR_data_ssl/fold_0 (5 validation
# writers, 1,440 pair records - the smallest proxy sample of the three
# datasets): genuine-pair median 0.2794, negative-pair median 0.5293
# (exactly 50.0%/50.0% active fraction). Same margins as the local Cell
# A/B runs on this encoder - DO NOT change these without re-running that
# sweep - see docs/claude_response/downstream_supervised_learning_approach.md
# SS16/SS18/SS20 for the full history of why each dataset/encoder
# combination needs its own sweep.
MARGIN_M_COMBINED = 0.28
MARGIN_N_COMBINED = 0.53
SINKHORN_EPSILON = 0.05
SINKHORN_ITERATIONS = 50

# -- Verification (checkpoint-selection metric, Step 5 protocol) ----------------
VERIFICATION_NUM_REFERENCES = 8   # K=8, matches SURDS's own protocol
VERIFICATION_SEEDS: tuple[int, ...] = (101, 202, 303, 404, 505)
VERIFICATION_BATCH_SIZE = 64

# -- Optimizer --------------------------------------------------------------------
# BATCH_SIZE deliberately left at 8, matching every local run - BatchNorm
# inside the (now fully trainable) encoder is batch-size sensitive, so
# changing this would make the run not directly comparable to the
# Hindi/Bengali Cell C results or CEDAR's own local Cell A/B runs despite
# Kaggle's T4 having far more VRAM.
BATCH_SIZE = 8
LEARNING_RATE = 1e-4        # head (projector + local_projection) LR
ENCODER_LEARNING_RATE = 1e-5  # ~10x lower than head LR - every encoder stage already encodes 50 epochs of DenseCL pretraining
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50              # generous ceiling; PATIENCE is what actually ends the run
PERIODIC_SAVE_FREQUENCY = 1  # save every epoch's checkpoint, not just the best one
PATIENCE = 10
MIN_EPOCHS = 15

TRAIN_SEED = 42
VAL_TUPLES_PER_ANCHOR = 4
VAL_SEED = 314

# --- Dataloader ------------------------------------------------------------------
# Kaggle (Linux) uses fork() for DataLoader workers - cheaper/more reliable
# than Windows spawn(), so unlike the local runs this can safely use workers.
NUM_WORKERS = 2

# ============================================================================
# End of editable section.
# ============================================================================


In [ ]:
import sys

sys.path.insert(0, "/kaggle/working/supervised_src")


## 3. Held-out writer splits (test + SSL validation, fold_0 CEDAR)

The exact fold_0 CEDAR test/validation writer splits already used for
every local CEDAR run (Cell A frozen, Cell B stage4, this Cell C) -
written here verbatim, not regenerated, so held-out writers can never
silently drift.

In [ ]:
import json
import os

# The exact fold_0 held-out writer splits already used for every local
# CEDAR run (Cell A frozen, Cell B stage4, this Cell C) - written here
# verbatim (not regenerated) from D:\...\DenseCL_approach\data\
# {test_set_writer_split,validation_set_writer_split}\fold_0\CEDAR_*.json,
# so the held-out writers can never silently drift between the local runs
# and this Kaggle run. Only CEDAR is needed for this notebook.

TEST_SPLIT = {
    "dataset": "CEDAR",
    "total_writers": 55,
    "num_test_writers": 15,
    "seed": 42,
    "test_writer_ids": [
        "10", "11", "14", "15", "16", "17", "22", "23", "25", "34",
        "40", "43", "46", "49", "52"
    ]
}

VALIDATION_SPLIT = {
    "dataset": "CEDAR",
    "total_writers": 55,
    "num_test_writers_excluded": 15,
    "num_eligible_writers": 40,
    "num_validation_writers": 5,
    "seed": 42,
    "validation_writer_ids": [
        "12", "21", "30", "31", "33"
    ]
}

TEST_DIR = "/kaggle/working/supervised_src/data/test_set_writer_split/fold_0"
VAL_DIR = "/kaggle/working/supervised_src/data/validation_set_writer_split/fold_0"
os.makedirs(TEST_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

with open(os.path.join(TEST_DIR, "CEDAR_test_writers.json"), "w") as f:
    json.dump(TEST_SPLIT, f, indent=2)

with open(os.path.join(VAL_DIR, "CEDAR_validation_writers.json"), "w") as f:
    json.dump(VALIDATION_SPLIT, f, indent=2)

print(f"Wrote test split ({len(TEST_SPLIT['test_writer_ids'])} writers) -> {TEST_DIR}")
print(f"Wrote validation split ({len(VALIDATION_SPLIT['validation_writer_ids'])} writers) -> {VAL_DIR}")


## 4. Source modules

Each cell below writes one module, adapted from the local project's
already-verified `supervised_verification_approach/` code (flattened into
one directory - no `sys.path` bootstrapping needed - and re-pointed at
`kaggle_config.py` for paths). The actual training/loss/matching/model
logic is byte-for-byte identical to the local version, and every module
is fully dataset-agnostic (parameterized by `kaggle_config.DATASET_NAME`)
- unchanged from the Bengali Cell B/C notebooks.

In [ ]:
%%writefile /kaggle/working/supervised_src/preprocess.py
"""Deterministic preprocessing pipeline for offline signature images.

Pipeline: grayscale -> Otsu binarize -> crop to ink bounding box ->
square-pad -> resize to 256x256 -> Otsu again -> clean binary image.

The output is a 256x256 uint8 binary image with ink strokes at maximum
intensity (255) against a uniform black (0) background, matching the
preprocessing described for the reconstruction-SSL baseline (Fig 3.2 of
the thesis report) and reused unchanged for the DenseCL pretext.
"""

from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

# Silences OpenCV's own internal WARNING-level logging (e.g. grfmt_tiff.cpp's
# "TIFFFetchNormalTag: ... Software tag contains null byte" spam - harmless
# metadata truncation on every BHSig260 .tif read, not a real problem) while
# still surfacing actual errors. Set once at import time since this is a
# process-wide OpenCV setting, not per-call. `dataset.py` already sets this
# for the SSL-side pipeline; set here too since this module (not dataset.py)
# is the one every cv2.imread caller across both SSL and downstream
# supervised code actually imports.
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

TARGET_SIZE = 256


def otsu_binarize(gray: np.ndarray) -> np.ndarray:
    """Otsu threshold with inversion so ink pixels become the foreground (255).

    Public because `augment.py` reuses this exact step to re-binarize every
    augmented view after geometric/intensity perturbations, keeping the
    "always clean binary in, clean binary out" invariant throughout.
    """
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary


def _crop_to_ink_bbox(gray: np.ndarray, ink_mask: np.ndarray) -> np.ndarray:
    """Crop `gray` to the tight bounding box of the non-zero pixels in `ink_mask`."""
    ys, xs = np.nonzero(ink_mask)
    if ys.size == 0 or xs.size == 0:
        # No ink detected (blank/corrupt scan) - fall back to the full image
        # rather than crashing on an empty crop.
        return gray
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return gray[y0:y1, x0:x1]


def _square_pad(gray: np.ndarray, fill_value: int = 255) -> np.ndarray:
    """Center `gray` on a square canvas, padding with `fill_value` (white)."""
    h, w = gray.shape
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    return cv2.copyMakeBorder(
        gray, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=fill_value,
    )


def preprocess_signature(gray: np.ndarray, target_size: int = TARGET_SIZE) -> np.ndarray:
    """Run the full deterministic preprocessing pipeline on one signature image.

    Parameters
    ----------
    gray:
        Single-channel (grayscale) signature image, as returned by
        ``cv2.imread(path, cv2.IMREAD_GRAYSCALE)``.
    target_size:
        Side length of the final square output image.

    Returns
    -------
    np.ndarray
        A ``target_size x target_size`` uint8 binary image with ink
        strokes at maximum intensity (255) against a black (0) background.
    """
    if gray.ndim != 2:
        raise ValueError(f"Expected a single-channel grayscale image, got shape {gray.shape}")

    # Step 1: locate ink strokes via Otsu binarization. This mask is used
    # only to find the crop region and is discarded afterwards.
    ink_mask = otsu_binarize(gray)

    # Step 2: crop to the tight bounding box around the ink so writer- and
    # scanner-dependent margins don't affect downstream scale.
    cropped = _crop_to_ink_bbox(gray, ink_mask)

    # Step 3: pad to a square canvas (centered) to preserve aspect ratio
    # and avoid the anisotropic distortion a direct resize would introduce.
    squared = _square_pad(cropped, fill_value=255)

    # Step 4: resize to the fixed resolution the encoder consumes.
    resized = cv2.resize(
        squared, (target_size, target_size), interpolation=cv2.INTER_AREA
    )

    # Step 5: re-binarize to remove grayscale interpolation artifacts
    # introduced by resizing, yielding a clean binary signature.
    clean_binary = otsu_binarize(resized)

    return clean_binary


def load_and_preprocess(path: str | Path) -> tuple[np.ndarray, np.ndarray]:
    """Load an image from disk and return `(original_grayscale, preprocessed)`."""
    path = Path(path)
    original_gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if original_gray is None:
        raise FileNotFoundError(f"Could not read image at {path}")
    preprocessed = preprocess_signature(original_gray)
    return original_gray, preprocessed


if __name__ == "__main__":
    import sys

    if len(sys.argv) != 2:
        print("Usage: python preprocess.py <path-to-signature-image>")
        raise SystemExit(1)

    orig, clean = load_and_preprocess(sys.argv[1])
    print(f"Original shape:     {orig.shape}")
    print(f"Preprocessed shape: {clean.shape}, dtype={clean.dtype}, "
          f"unique values={np.unique(clean)}")


In [ ]:
%%writefile /kaggle/working/supervised_src/mask.py
"""Foreground (ink) mask computation for the encoder's dense feature grid.

The dense DenseCL correspondence loss operates on the encoder's 32x32
feature grid, not on raw pixels, so the pixel-level binary mask already
present in a preprocessed signature (ink=255, background=0) has to be
summarized down to one "how much ink is in this cell" number per grid
cell, and then turned into a foreground/background decision per cell.
"""

from __future__ import annotations

import numpy as np

GRID_SIZE = 32


def compute_ink_coverage(binary_image: np.ndarray, grid_size: int = GRID_SIZE) -> np.ndarray:
    """Fraction of ink pixels in each non-overlapping grid cell.

    Parameters
    ----------
    binary_image:
        A square binary image (e.g. the output of `preprocess_signature`),
        ink pixels > 0, background == 0.
    grid_size:
        Number of cells per side. Must evenly divide the image side length
        (32 for a 256x256 image, matching the encoder's three stride-2
        downsamples: 256 / 8 = 32).

    Returns
    -------
    np.ndarray
        `(grid_size, grid_size)` float32 array; each value is the fraction
        (0.0-1.0) of ink pixels inside that cell's pixel block.
    """
    h, w = binary_image.shape
    if h != w:
        raise ValueError(f"Expected a square image, got shape {binary_image.shape}")
    if h % grid_size != 0:
        raise ValueError(
            f"Image side length {h} is not evenly divisible by grid_size {grid_size}"
        )

    cell = h // grid_size
    ink = (binary_image > 0).astype(np.float32)
    coverage = ink.reshape(grid_size, cell, grid_size, cell).mean(axis=(1, 3))
    return coverage


def foreground_mask_from_coverage(coverage: np.ndarray, min_coverage: float = 0.0) -> np.ndarray:
    """Turn per-cell ink coverage into a foreground/background decision.

    A cell counts as foreground if its ink coverage is strictly greater
    than `min_coverage`. The default (0.0) means "any ink pixel in the
    cell counts" - the grid-cell equivalent of the pixel-level rule
    already used for the reconstruction pretext's foreground-weighted MSE
    (Eq. 3.5 of the thesis report: w(p) = w_fg if t(p) > 0 else w_bg).
    """
    return coverage > min_coverage


def compute_foreground_mask(
    binary_image: np.ndarray, grid_size: int = GRID_SIZE, min_coverage: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    """Convenience wrapper: binary image -> (foreground_mask, ink_coverage)."""
    coverage = compute_ink_coverage(binary_image, grid_size=grid_size)
    mask = foreground_mask_from_coverage(coverage, min_coverage=min_coverage)
    return mask, coverage


if __name__ == "__main__":
    import sys

    from preprocess import load_and_preprocess

    if len(sys.argv) != 2:
        print("Usage: python mask.py <path-to-signature-image>")
        raise SystemExit(1)

    _, preprocessed = load_and_preprocess(sys.argv[1])
    fg_mask, coverage = compute_foreground_mask(preprocessed)
    print(f"Grid shape: {fg_mask.shape}")
    print(f"Foreground cells: {fg_mask.sum()} / {fg_mask.size} "
          f"({100 * fg_mask.sum() / fg_mask.size:.1f}%)")
    print(f"Coverage min/mean/max: {coverage.min():.3f}/{coverage.mean():.3f}/{coverage.max():.3f}")


In [ ]:
%%writefile /kaggle/working/supervised_src/encoder.py
"""ResNet-style backbone - faithful port of the original thesis encoder.

Ported from `Thesis_Final/ssl_pretraining/models/Encoder.py` and
`Thesis_Final/ssl_pretraining/models/encoder/*.py` (the code that actually
produced the report's numbers - `SelfSupervisedNetwork.__init__` there
instantiates `Encoder(norm_type=norm_type)` with every other argument left
at its default, so the defaults below are not a guess, they're what was
used). This replaces an earlier version of this file that reconstructed the
architecture from the report's prose description alone (stage widths and
downsample placement only - it did not specify a stem or block layout), an
assumption flagged at the time and now resolved by reading the real source.

Two differences from that earlier guess, now corrected:
  - There is a dedicated 2-layer stem (plain `ConvNormAct`, no residual
    connection) that expands 1 -> 32 -> 32 channels before stage 1, not
    folded into stage 1 itself.
  - Stages 2-4 are each exactly one `ProjectionResidualBlock` (the
    stride-2 downsampling block) followed by `num_identity_blocks`
    `IdentityResidualBlock`s (1 by default) - not a generic "first block
    strided, rest identical" pattern with an arbitrary block count.

Architecture, in order: stem (1->32->32, no downsample) -> stage1 (2
identity blocks @ 32, no downsample) -> stage2 (32->64, downsample) ->
stage3 (64->128, downsample) -> stage4 (128->256, downsample). Three
downsamples total, matching report Fig 3.3 (256x256 input -> 32x32x256
dense feature grid), and the same 32x32 resolution `mask.py`'s foreground
grid is computed at.
"""

from __future__ import annotations

import torch
import torch.nn as nn

TARGET_SIZE = 256  # must match preprocess.TARGET_SIZE for the 32x32 grid to hold


def build_norm_layer(num_features: int, norm_type: str = "batch") -> nn.Module:
    if norm_type == "batch":
        return nn.BatchNorm2d(num_features)
    if norm_type == "instance":
        return nn.InstanceNorm2d(num_features, affine=True)
    raise ValueError(f"Unsupported norm_type: {norm_type}")


class ConvNormAct(nn.Module):
    """Conv -> norm -> activation. Used only by the stem (no residual connection)."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int | None = None,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        if padding is None:
            padding = kernel_size // 2

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            build_norm_layer(out_channels, norm_type=norm_type),
            activation_layer(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SignatureStem(nn.Module):
    """Two plain ConvNormAct layers, no downsample: 1 -> 32 -> 32 channels by default."""

    def __init__(
        self,
        in_channels: int = 1,
        stem_channels: tuple[int, ...] = (32, 32),
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        blocks = []
        current_in = in_channels
        for current_out in stem_channels:
            blocks.append(
                ConvNormAct(current_in, current_out, kernel_size=3, stride=1,
                             norm_type=norm_type, activation_layer=activation_layer)
            )
            current_in = current_out

        self.layers = nn.Sequential(*blocks)
        self.out_channels = stem_channels[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class IdentityResidualBlock(nn.Module):
    """Two 3x3 convs at a fixed channel count, plain identity skip (no projection)."""

    def __init__(self, in_channels: int, norm_type: str = "batch", activation_layer: type[nn.Module] = nn.ReLU) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm1 = build_norm_layer(in_channels, norm_type=norm_type)
        self.act1 = activation_layer(inplace=True)

        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = build_norm_layer(in_channels, norm_type=norm_type)

        self.out_act = activation_layer(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.act1(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.out_act(out + residual)


class ProjectionResidualBlock(nn.Module):
    """Stride-2, channel-changing residual block: 1x1-conv shortcut, two 3x3 convs on the main path."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 2,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = build_norm_layer(out_channels, norm_type=norm_type)
        self.act1 = activation_layer(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = build_norm_layer(out_channels, norm_type=norm_type)

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            build_norm_layer(out_channels, norm_type=norm_type),
        )
        self.out_act = activation_layer(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out = self.act1(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.out_act(out + residual)


class ResidualStage(nn.Module):
    """Stage 1: `num_blocks` IdentityResidualBlocks at a fixed channel count, no downsample."""

    def __init__(self, channels: int, num_blocks: int = 2, norm_type: str = "batch", activation_layer: type[nn.Module] = nn.ReLU) -> None:
        super().__init__()
        self.blocks = nn.Sequential(*[
            IdentityResidualBlock(channels, norm_type=norm_type, activation_layer=activation_layer)
            for _ in range(num_blocks)
        ])
        self.out_channels = channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)


class TransitionResidualStage(nn.Module):
    """Stages 2-4: one ProjectionResidualBlock (downsample + channel change) then `num_identity_blocks` IdentityResidualBlocks."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 2,
        num_identity_blocks: int = 1,
        norm_type: str = "batch",
        activation_layer: type[nn.Module] = nn.ReLU,
    ) -> None:
        super().__init__()
        blocks = [ProjectionResidualBlock(in_channels, out_channels, stride=stride,
                                           norm_type=norm_type, activation_layer=activation_layer)]
        for _ in range(num_identity_blocks):
            blocks.append(IdentityResidualBlock(out_channels, norm_type=norm_type, activation_layer=activation_layer))

        self.blocks = nn.Sequential(*blocks)
        self.out_channels = out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)


class Encoder(nn.Module):
    """The full backbone: stem -> stage1 -> stage2 -> stage3 -> stage4.

    `forward(x, pool)`: `pool=False` returns the dense `(batch, 256, 32, 32)`
    feature grid (what the DenseCL dense head and the foreground masks need);
    `pool=True` returns a globally-averaged `(batch, 256)` vector (what a
    global head - or the existing thesis's frozen-encoder downstream stage -
    needs). Matches the original `Encoder.forward` interface exactly, so
    this module is a drop-in match for how the thesis's own code calls it.
    """

    def __init__(
        self,
        in_channels: int = 1,
        stem_channels: tuple[int, ...] = (32, 32),
        stage1_blocks: int = 2,
        stage2_out_channels: int = 64,
        stage2_identity_blocks: int = 1,
        stage3_out_channels: int = 128,
        stage3_identity_blocks: int = 1,
        stage4_out_channels: int = 256,
        stage4_identity_blocks: int = 1,
        norm_type: str = "batch",
    ) -> None:
        super().__init__()

        self.stem = SignatureStem(in_channels=in_channels, stem_channels=stem_channels, norm_type=norm_type)

        self.stage1 = ResidualStage(channels=self.stem.out_channels, num_blocks=stage1_blocks, norm_type=norm_type)

        self.stage2 = TransitionResidualStage(
            in_channels=self.stage1.out_channels, out_channels=stage2_out_channels,
            stride=2, num_identity_blocks=stage2_identity_blocks, norm_type=norm_type,
        )
        self.stage3 = TransitionResidualStage(
            in_channels=self.stage2.out_channels, out_channels=stage3_out_channels,
            stride=2, num_identity_blocks=stage3_identity_blocks, norm_type=norm_type,
        )
        self.stage4 = TransitionResidualStage(
            in_channels=self.stage3.out_channels, out_channels=stage4_out_channels,
            stride=2, num_identity_blocks=stage4_identity_blocks, norm_type=norm_type,
        )

        self.out_channels = self.stage4.out_channels
        self.feature_dim = self.out_channels  # alias used elsewhere in this project (dataset.py, momentum.py)

    def forward(self, x: torch.Tensor, pool: bool = False) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f"Expected input shape (batch, channels, H, W), got {tuple(x.shape)}")
        out = self.stem(x)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        if pool:
            out = torch.mean(out, dim=(2, 3))
        return out


if __name__ == "__main__":
    encoder = Encoder()
    dummy = torch.zeros(2, 1, TARGET_SIZE, TARGET_SIZE)

    dense = encoder(dummy, pool=False)
    pooled = encoder(dummy, pool=True)

    num_params = sum(p.numel() for p in encoder.parameters())
    expected_grid = TARGET_SIZE // 8  # three stride-2 downsamples

    print(f"Input shape:  {tuple(dummy.shape)}")
    print(f"Dense output shape:  {tuple(dense.shape)} (expected grid size {expected_grid}x{expected_grid})")
    print(f"Pooled output shape: {tuple(pooled.shape)}")
    print(f"Parameter count: {num_params:,}")


In [ ]:
%%writefile /kaggle/working/supervised_src/test_set_creation.py
"""Writer-level test-set split lookup - Kaggle version.

Trimmed to only what writer_splits.py needs (list_writer_ids,
load_test_writer_ids) - the split itself is NOT regenerated here, the
notebook writes the already-frozen fold_0 split JSON directly to
OUTPUT_DIR in an earlier cell (see "Held-out writer splits"), so the
random-selection machinery the local project's version keeps for
completeness is omitted.
"""

from __future__ import annotations

import json
from pathlib import Path

from kaggle_config import DATA_ROOT

SEED = 42
OUTPUT_DIR = Path("/kaggle/working/supervised_src/data/test_set_writer_split")


def list_writer_ids(dataset_dir: Path) -> list[str]:
    """Every writer directory name under a dataset folder, as strings."""
    return sorted(p.name for p in dataset_dir.iterdir() if p.is_dir())


def load_test_writer_ids(dataset_name: str, split_dir: Path = OUTPUT_DIR) -> set[str]:
    """Read back a previously-created split. Returns an empty set for a
    dataset with no split file, meaning "hold out nothing" - the correct
    behavior for a dataset that has none."""
    split_path = split_dir / f"{dataset_name}_test_writers.json"
    if not split_path.exists():
        return set()
    with open(split_path) as f:
        return set(json.load(f)["test_writer_ids"])


In [ ]:
%%writefile /kaggle/working/supervised_src/validation_set_creation.py
"""Writer-level SSL validation split lookup - Kaggle version.

Companion to test_set_creation.py, trimmed the same way - the split is
written verbatim by an earlier notebook cell, not regenerated here.
"""

from __future__ import annotations

import json
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/supervised_src/data/validation_set_writer_split")


def load_validation_writer_ids(dataset_name: str, split_dir: Path = OUTPUT_DIR) -> set[str]:
    """Read back a previously-created split. Returns an empty set if no
    split file exists for this dataset yet."""
    split_path = split_dir / f"{dataset_name}_validation_writers.json"
    if not split_path.exists():
        return set()
    with open(split_path) as f:
        return set(json.load(f)["validation_writer_ids"])


In [ ]:
%%writefile /kaggle/working/supervised_src/writer_splits.py
"""Per-dataset writer split for the downstream supervised verification
phase - Kaggle version. Same logic as the local project's writer_splits.py,
re-pointed at kaggle_config.py's DATA_ROOT and the flat Kaggle source
directory's split JSON locations instead of path-derived ones.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from kaggle_config import DATA_ROOT
from test_set_creation import OUTPUT_DIR as TEST_SPLIT_ROOT
from test_set_creation import list_writer_ids, load_test_writer_ids
from validation_set_creation import OUTPUT_DIR as VALIDATION_SPLIT_ROOT
from validation_set_creation import load_validation_writer_ids

SUPPORTED_DATASETS: tuple[str, ...] = ("CEDAR", "BHSig260_Bengali", "BHSig260_Hindi")


@dataclass(frozen=True)
class WriterSplit:
    dataset: str
    train_writer_ids: list[str]
    validation_writer_ids: list[str]
    test_writer_ids: list[str]

    @property
    def total_writers(self) -> int:
        return len(self.train_writer_ids) + len(self.validation_writer_ids) + len(self.test_writer_ids)


def get_writer_split(dataset_name: str, fold: str | None = None) -> WriterSplit:
    if dataset_name not in SUPPORTED_DATASETS:
        raise ValueError(f"'{dataset_name}' is not supported. Supported datasets: {SUPPORTED_DATASETS}")

    dataset_dir = DATA_ROOT / dataset_name
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_dir}")

    test_split_dir = TEST_SPLIT_ROOT / fold if fold else TEST_SPLIT_ROOT
    validation_split_dir = VALIDATION_SPLIT_ROOT / fold if fold else VALIDATION_SPLIT_ROOT

    all_writer_ids = set(list_writer_ids(dataset_dir))
    test_writer_ids = load_test_writer_ids(dataset_name, split_dir=test_split_dir)
    validation_writer_ids = load_validation_writer_ids(dataset_name, split_dir=validation_split_dir)

    if not test_writer_ids:
        raise ValueError(f"No test split found for '{dataset_name}' under fold {fold!r}.")
    if not validation_writer_ids:
        raise ValueError(f"No validation split found for '{dataset_name}' under fold {fold!r}.")

    overlap = test_writer_ids & validation_writer_ids
    if overlap:
        raise ValueError(f"'{dataset_name}' has writer(s) in BOTH test and validation splits: {sorted(overlap, key=int)}")

    unknown_test = test_writer_ids - all_writer_ids
    unknown_validation = validation_writer_ids - all_writer_ids
    if unknown_test or unknown_validation:
        raise ValueError(
            f"'{dataset_name}' split file(s) reference writer(s) not found on disk under "
            f"{dataset_dir}: test={sorted(unknown_test, key=int)}, validation={sorted(unknown_validation, key=int)}"
        )

    train_writer_ids = all_writer_ids - test_writer_ids - validation_writer_ids

    return WriterSplit(
        dataset=dataset_name,
        train_writer_ids=sorted(train_writer_ids, key=int),
        validation_writer_ids=sorted(validation_writer_ids, key=int),
        test_writer_ids=sorted(test_writer_ids, key=int),
    )


In [ ]:
%%writefile /kaggle/working/supervised_src/dual_triplet_dataset.py
"""4-tuple dataset for downstream dual-triplet supervised verification
training - Kaggle version (flat imports, no sys.path bootstrapping needed
since every module already lives in one directory on sys.path).

Uses this project's preprocessing (`preprocess_signature`) and tensor
convention (binary {0,1} float32) - the SSL-pretrained encoder was trained
on exactly this input convention (self_supervised_approach's dataset.py,
`(binary_view > 0).astype(np.float32)`).
"""

from __future__ import annotations

import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from preprocess import preprocess_signature

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
LABELED_DATASETS = ("CEDAR", "BHSig260_Bengali", "BHSig260_Hindi")


def list_genuine_and_forged_images(writer_dir: Path, dataset_name: str) -> tuple[list[Path], list[Path]]:
    """Genuine and forged signature paths for one writer.
      - CEDAR: `original_<writer>_<n>.png` (genuine) vs `forgeries_...` (forged)
      - BHSig260 (Bengali/Hindi): `*-G-*.tif` (genuine) vs `*-F-*.tif` (forged)
    """
    if dataset_name not in LABELED_DATASETS:
        raise ValueError(f"Unsupported dataset for genuine/forged labeling: {dataset_name}")

    all_images = sorted(p for p in writer_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    if dataset_name == "CEDAR":
        genuine = [p for p in all_images if p.name.lower().startswith("original_")]
        forged = [p for p in all_images if p.name.lower().startswith("forgeries_")]
    else:
        genuine = [p for p in all_images if "-g-" in p.name.lower()]
        forged = [p for p in all_images if "-f-" in p.name.lower()]
    return genuine, forged


def load_signature_tensor(image_path: Path) -> torch.Tensor:
    """Signature file -> preprocessed 256x256 view -> binary {0,1} float32
    tensor, shape (1, 256, 256)."""
    raw = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if raw is None:
        raise FileNotFoundError(f"Could not read image at {image_path}")
    view = preprocess_signature(raw)
    binary = (view > 0).astype(np.float32)
    return torch.from_numpy(binary).unsqueeze(0)


class DualTripletDataset(Dataset):
    """Emits 4-tuples: (anchor_genuine, positive_genuine, negative_intra_forgery, negative_inter_genuine).

    One anchor record exists per genuine signature in the writer pool, so
    `len(dataset)` = total genuine count and one epoch uses every genuine
    once as an anchor. Call `set_epoch()` before each epoch so the random
    draw of the other three tuple members changes epoch to epoch.
    """

    def __init__(
        self,
        writer_ids: list[str],
        dataset_dir: Path,
        dataset_name: str,
        seed: int = 42,
    ) -> None:
        super().__init__()
        self.seed = int(seed)
        self.current_epoch = 0

        self.writer_to_genuine_paths: dict[str, list[Path]] = {}
        self.writer_to_forgery_paths: dict[str, list[Path]] = {}
        for writer_id in writer_ids:
            writer_dir = dataset_dir / writer_id
            if not writer_dir.is_dir():
                raise FileNotFoundError(f"Writer directory not found: {writer_dir}")

            genuine_paths, forgery_paths = list_genuine_and_forged_images(writer_dir, dataset_name)
            if len(genuine_paths) < 2:
                raise ValueError(
                    f"Writer {writer_id} has fewer than two genuine signatures "
                    f"(found {len(genuine_paths)}). Cannot form anchor-positive pairs."
                )
            if len(forgery_paths) < 1:
                raise ValueError(f"Writer {writer_id} has no forgery signatures.")

            self.writer_to_genuine_paths[writer_id] = genuine_paths
            self.writer_to_forgery_paths[writer_id] = forgery_paths

        self.writer_ids: list[str] = sorted(self.writer_to_genuine_paths.keys(), key=int)
        if len(self.writer_ids) < 2:
            raise ValueError(
                f"Need at least two writers for inter-negative sampling (found {len(self.writer_ids)})."
            )

        self.anchor_records: list[tuple[str, Path]] = [
            (writer_id, path)
            for writer_id in self.writer_ids
            for path in self.writer_to_genuine_paths[writer_id]
        ]

    def __len__(self) -> int:
        return len(self.anchor_records)

    def set_epoch(self, epoch: int) -> None:
        self.current_epoch = int(epoch)

    def __getitem__(self, index: int) -> dict:
        anchor_writer_id, anchor_path = self.anchor_records[index]
        rng = random.Random(self.seed + (self.current_epoch * 100003) + index)

        positive_candidates = [p for p in self.writer_to_genuine_paths[anchor_writer_id] if p != anchor_path]
        positive_path = rng.choice(positive_candidates)

        negative_intra_path = rng.choice(self.writer_to_forgery_paths[anchor_writer_id])

        inter_writer_id = rng.choice([wid for wid in self.writer_ids if wid != anchor_writer_id])
        negative_inter_path = rng.choice(self.writer_to_genuine_paths[inter_writer_id])

        return {
            "anchor": load_signature_tensor(anchor_path),
            "positive": load_signature_tensor(positive_path),
            "negative_intra": load_signature_tensor(negative_intra_path),
            "negative_inter": load_signature_tensor(negative_inter_path),
            "anchor_writer_id": anchor_writer_id,
            "inter_writer_id": inter_writer_id,
        }


In [ ]:
%%writefile /kaggle/working/supervised_src/fixed_dual_triplet_dataset.py
"""Fixed 4-tuple dataset for downstream dual-triplet VALIDATION loss
tracking - Kaggle version (flat imports).

Unlike `DualTripletDataset` (training), the 4-tuples here are built ONCE
with a fixed seed and never change between epochs, so a rise or fall in
validation loss can only mean the model changed.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path

from torch.utils.data import Dataset

from dual_triplet_dataset import load_signature_tensor, list_genuine_and_forged_images

DEFAULT_TUPLES_PER_ANCHOR = 4
DEFAULT_SEED = 314


@dataclass(frozen=True)
class FixedTripletRecord:
    anchor_writer_id: str
    inter_writer_id: str
    anchor_path: Path
    positive_path: Path
    negative_intra_path: Path
    negative_inter_path: Path


def build_fixed_triplet_records(
    writer_ids: list[str],
    dataset_dir: Path,
    dataset_name: str,
    tuples_per_anchor: int = DEFAULT_TUPLES_PER_ANCHOR,
    seed: int = DEFAULT_SEED,
) -> list[FixedTripletRecord]:
    """Deterministically build `tuples_per_anchor` fixed 4-tuples per
    genuine anchor. Calling this twice with the same arguments returns an
    identical list."""
    writer_to_genuine_paths: dict[str, list[Path]] = {}
    writer_to_forgery_paths: dict[str, list[Path]] = {}
    for writer_id in writer_ids:
        writer_dir = dataset_dir / writer_id
        if not writer_dir.is_dir():
            raise FileNotFoundError(f"Writer directory not found: {writer_dir}")

        genuine_paths, forgery_paths = list_genuine_and_forged_images(writer_dir, dataset_name)
        if len(genuine_paths) < 2:
            raise ValueError(
                f"Writer {writer_id} has fewer than two genuine signatures "
                f"(found {len(genuine_paths)}). Cannot form anchor-positive pairs."
            )
        if len(forgery_paths) < 1:
            raise ValueError(f"Writer {writer_id} has no forgery signatures.")

        writer_to_genuine_paths[writer_id] = genuine_paths
        writer_to_forgery_paths[writer_id] = forgery_paths

    sorted_writer_ids = sorted(writer_to_genuine_paths.keys(), key=int)
    if len(sorted_writer_ids) < 2:
        raise ValueError(
            f"Need at least two writers for inter-negative sampling (found {len(sorted_writer_ids)})."
        )

    records: list[FixedTripletRecord] = []
    for writer_id in sorted_writer_ids:
        inter_writer_candidates = [w for w in sorted_writer_ids if w != writer_id]
        genuine_paths = writer_to_genuine_paths[writer_id]
        forgery_paths = writer_to_forgery_paths[writer_id]

        for anchor_index, anchor_path in enumerate(genuine_paths):
            rng = random.Random(seed + (int(writer_id) * 1009) + anchor_index)
            positive_candidates = [p for p in genuine_paths if p != anchor_path]

            for tuple_index in range(tuples_per_anchor):
                positive_path = positive_candidates[tuple_index % len(positive_candidates)]
                negative_intra_path = forgery_paths[tuple_index % len(forgery_paths)]

                inter_writer_id = inter_writer_candidates[rng.randrange(len(inter_writer_candidates))]
                inter_genuine_paths = writer_to_genuine_paths[inter_writer_id]
                negative_inter_path = inter_genuine_paths[tuple_index % len(inter_genuine_paths)]

                records.append(FixedTripletRecord(
                    anchor_writer_id=writer_id,
                    inter_writer_id=inter_writer_id,
                    anchor_path=anchor_path,
                    positive_path=positive_path,
                    negative_intra_path=negative_intra_path,
                    negative_inter_path=negative_inter_path,
                ))

    return records


class FixedDualTripletDataset(Dataset):
    """Validation/test dataset backed by a pre-built, deterministic list of
    4-tuple records. Unlike `DualTripletDataset`, there is no `set_epoch()`."""

    def __init__(self, records: list[FixedTripletRecord]) -> None:
        super().__init__()
        if not records:
            raise ValueError("records is empty. Cannot build FixedDualTripletDataset.")
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        record = self.records[index]
        return {
            "anchor": load_signature_tensor(record.anchor_path),
            "positive": load_signature_tensor(record.positive_path),
            "negative_intra": load_signature_tensor(record.negative_intra_path),
            "negative_inter": load_signature_tensor(record.negative_inter_path),
            "anchor_writer_id": record.anchor_writer_id,
            "inter_writer_id": record.inter_writer_id,
        }


In [ ]:
%%writefile /kaggle/working/supervised_src/pair_dataset.py
"""Pair dataset for downstream double-margin supervised training - Kaggle
version (flat imports).

Wraps `DualTripletDataset` - one already-sampled 4-tuple (anchor, positive,
negative_intra, negative_inter) decomposes into exactly the three labeled
pairs a double-margin contrastive loss needs:

    (anchor, positive,       label=1)  # same writer, both genuine
    (anchor, negative_intra, label=0)  # same writer, forged
    (anchor, negative_inter, label=0)  # different writer, genuine
"""

from __future__ import annotations

from torch.utils.data import Dataset

from dual_triplet_dataset import DualTripletDataset

_PAIR_OTHER_KEYS = ("positive", "negative_intra", "negative_inter")
_PAIR_LABELS = (1.0, 0.0, 0.0)


class PairDataset(Dataset):
    def __init__(self, quad_dataset: DualTripletDataset) -> None:
        super().__init__()
        self.quad_dataset = quad_dataset

    def __len__(self) -> int:
        return len(self.quad_dataset) * 3

    def set_epoch(self, epoch: int) -> None:
        self.quad_dataset.set_epoch(epoch)

    def __getitem__(self, index: int) -> dict:
        quad_index, pair_slot = divmod(index, 3)
        quad = self.quad_dataset[quad_index]
        other_key = _PAIR_OTHER_KEYS[pair_slot]
        return {
            "image_a": quad["anchor"],
            "image_b": quad[other_key],
            "label": _PAIR_LABELS[pair_slot],
            "anchor_writer_id": quad["anchor_writer_id"],
        }


In [ ]:
%%writefile /kaggle/working/supervised_src/embedding_model.py
"""Downstream verification embedding model: the SSL-pretrained encoder plus
a small trainable projector - Kaggle version (flat imports).

`find_encoder_checkpoint` is inlined here (byte-for-byte identical logic to
the local project's `self_supervised_approach/analyzer/similarity_analyzer.
find_encoder_checkpoint`) rather than imported, since that module pulls in
a long chain of analysis-only dependencies (correspondence.py, visualize.py,
orchestration.py, augment.py) that this training notebook has no other use
for - only this one small function is actually needed here.
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from encoder import Encoder
from kaggle_config import SSL_RESULTS_ROOT


def find_encoder_checkpoint(run_dir: Path, epoch: Optional[int]) -> tuple[Path, int]:
    """Locate the encoder-only checkpoint to load. `epoch=None` picks the
    highest-numbered `encoder_epoch<N>.pt` found (the most-trained one)."""
    checkpoint_dir = run_dir / "checkpoints"
    if epoch is not None:
        path = checkpoint_dir / f"encoder_epoch{epoch}.pt"
        if not path.exists():
            raise FileNotFoundError(f"No checkpoint at {path}")
        return path, epoch

    candidates = sorted(checkpoint_dir.glob("encoder_epoch*.pt"))
    if not candidates:
        raise FileNotFoundError(f"No encoder checkpoints found under {checkpoint_dir}")

    def epoch_of(path: Path) -> int:
        return int(path.stem.replace("encoder_epoch", ""))

    best = max(candidates, key=epoch_of)
    return best, epoch_of(best)


class DownstreamVerificationModel(nn.Module):
    """Wraps the SSL-pretrained encoder for the downstream verification
    phase. Exposes two forward paths off the SAME shared encoder:

      - `forward_global(x)`: Method A - pooled 256-d backbone feature ->
        trainable projector -> L2-normalized embedding.
      - `forward_dense(x)`: Method B - raw dense backbone feature grid
        (`pool=False`), NO projector applied.
    """

    def __init__(
        self,
        pretrained_ssl_checkpoint_path: Optional[str | Path],
        trainable_encoder_stages: tuple[str, ...] = (),
        projector_hidden_dim: int = 256,
        embedding_dim: int = 256,
        norm_type: str = "batch",
        local_embedding_dim: Optional[int] = None,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(norm_type=norm_type)

        self.projector = nn.Sequential(
            nn.Linear(self.encoder.out_channels, projector_hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(projector_hidden_dim, embedding_dim),
        )

        self.local_projection: Optional[nn.Linear] = (
            nn.Linear(self.encoder.out_channels, local_embedding_dim)
            if local_embedding_dim is not None else None
        )

        if pretrained_ssl_checkpoint_path is not None:
            self._load_pretrained_encoder(Path(pretrained_ssl_checkpoint_path))
        else:
            print("No SSL checkpoint provided - encoder left at random initialization.")

        self._configure_partial_finetuning(trainable_encoder_stages)

    def _load_pretrained_encoder(self, checkpoint_path: Path) -> None:
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        missing_keys, unexpected_keys = self.encoder.load_state_dict(state_dict, strict=True)
        print(f"Loaded pretrained SSL encoder from: {checkpoint_path}")
        print(f"Missing keys:    {len(missing_keys)}")
        print(f"Unexpected keys: {len(unexpected_keys)}")

    def _configure_partial_finetuning(self, trainable_encoder_stages: tuple[str, ...]) -> None:
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        for stage_name in trainable_encoder_stages:
            if not hasattr(self.encoder, stage_name):
                raise ValueError(f"Unknown encoder stage: {stage_name}")
            for parameter in getattr(self.encoder, stage_name).parameters():
                parameter.requires_grad = True

        frozen_params = sum(p.numel() for p in self.encoder.parameters() if not p.requires_grad)
        trainable_params = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        projector_params = sum(p.numel() for p in self.projector.parameters())

        print(f"Trainable encoder stages:   {trainable_encoder_stages if trainable_encoder_stages else 'none (fully frozen)'}")
        print(f"Frozen encoder params:      {frozen_params:,}")
        print(f"Trainable encoder params:   {trainable_params:,}")
        print(f"Trainable projector params: {projector_params:,}")
        if self.local_projection is not None:
            local_params = sum(p.numel() for p in self.local_projection.parameters())
            print(f"Trainable local projection params: {local_params:,} (dim={self.local_projection.out_features})")

    def forward_global(self, image_tensor: torch.Tensor) -> torch.Tensor:
        pooled = self.encoder(image_tensor, pool=True)
        projected = self.projector(pooled)
        return F.normalize(projected, p=2, dim=1)

    def forward_dense(self, image_tensor: torch.Tensor) -> torch.Tensor:
        dense = self.encoder(image_tensor, pool=False)  # (B, 256, grid, grid)
        return dense.permute(0, 2, 3, 1)  # (B, grid, grid, 256)

    def forward(self, image_tensor: torch.Tensor) -> torch.Tensor:
        return self.forward_global(image_tensor)


def load_downstream_model(
    run_name: str,
    checkpoint_epoch: Optional[int],
    device: torch.device,
    trainable_encoder_stages: tuple[str, ...] = (),
    projector_hidden_dim: int = 256,
    embedding_dim: int = 256,
    norm_type: str = "batch",
    local_embedding_dim: Optional[int] = None,
) -> DownstreamVerificationModel:
    """Locate `encoder_epoch<N>.pt` for `run_name` under
    `kaggle_config.SSL_RESULTS_ROOT`, build the model, move to `device`."""
    checkpoint_path, epoch = find_encoder_checkpoint(SSL_RESULTS_ROOT / run_name, checkpoint_epoch)
    model = DownstreamVerificationModel(
        pretrained_ssl_checkpoint_path=checkpoint_path,
        trainable_encoder_stages=trainable_encoder_stages,
        projector_hidden_dim=projector_hidden_dim,
        embedding_dim=embedding_dim,
        norm_type=norm_type,
        local_embedding_dim=local_embedding_dim,
    )
    print(f"Run: {run_name} / epoch {epoch}")
    return model.to(device)


In [ ]:
%%writefile /kaggle/working/supervised_src/double_margin_distance_loss.py
"""Distance-based double-margin contrastive loss for Step 4's combined
distance - the precomputed-scalar-distance counterpart to
`double_margin_loss.DoubleMarginLoss` (which takes two embeddings and
computes `pairwise_distance` internally).

Needed because the combined distance (global + structural) is not a
simple embedding distance: the structural term only ever exists as a
PAIRWISE scalar produced by Sinkhorn matching two piles of pieces - there
is no single fixed-length "combined embedding" per image a generic
embedding-based loss could consume. Same relationship
`distance_triplet_loss.DistanceDualTripletLoss` already has to
`dual_triplet_loss.DualTripletLoss` in this codebase, just for the
pair/double-margin loss instead of the triplet one.

Same formula as `DoubleMarginLoss` (DetailSemNet Eq. 13):

    y * max(0, dist - m)^2 + (1 - y) * max(0, n - dist)^2

`m`/`n` here are NOT Step 2/3's 0.46/0.96 - those were swept on pure
global-embedding distance and do not transfer to the combined distance's
different composition/scale. See `training/sweep_margins_combined.py`.
"""

from __future__ import annotations

import torch
import torch.nn as nn


def _masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean of `values` where `mask` is True; 0.0 (not NaN) if no elements
    match - same rationale as `double_margin_loss._masked_mean`."""
    if mask.sum() == 0:
        return torch.zeros((), device=values.device)
    return values[mask].mean()


class DoubleMarginDistanceLoss(nn.Module):
    def __init__(self, margin_m: float, margin_n: float) -> None:
        super().__init__()
        if not margin_m < margin_n:
            raise ValueError(f"margin_m ({margin_m}) must be < margin_n ({margin_n})")
        self.margin_m = margin_m
        self.margin_n = margin_n

    def forward(self, distance: torch.Tensor, label: torch.Tensor) -> dict[str, torch.Tensor]:
        is_positive = label > 0.5

        positive_term = label * torch.clamp(distance - self.margin_m, min=0.0).pow(2)
        negative_term = (1.0 - label) * torch.clamp(self.margin_n - distance, min=0.0).pow(2)
        loss = (positive_term + negative_term).mean()

        with torch.no_grad():
            positive_active_rate = _masked_mean((distance > self.margin_m).float(), is_positive)
            negative_active_rate = _masked_mean((distance < self.margin_n).float(), ~is_positive)
            positive_distance_mean = _masked_mean(distance, is_positive)
            negative_distance_mean = _masked_mean(distance, ~is_positive)
            positive_loss_mean = _masked_mean(positive_term, is_positive)
            negative_loss_mean = _masked_mean(negative_term, ~is_positive)

        return {
            "loss": loss,
            "positive_loss": positive_loss_mean,
            "negative_loss": negative_loss_mean,
            "positive_distance_mean": positive_distance_mean,
            "negative_distance_mean": negative_distance_mean,
            "positive_active_rate": positive_active_rate,
            "negative_active_rate": negative_active_rate,
        }


In [ ]:
%%writefile /kaggle/working/supervised_src/dense_matching.py
"""Method B: piece-by-piece dense matching distance between two signatures
(entropic-regularized optimal transport, Sinkhorn) - Kaggle version (flat
imports). See the local project's matching/dense_matching.py for the full
design rationale - logic here is byte-for-byte identical.
"""

from __future__ import annotations

import torch
import torch.nn.functional as F

from mask import compute_foreground_mask


def compute_mask_from_tensor(image_tensor: torch.Tensor) -> torch.Tensor:
    image_np = image_tensor.squeeze().detach().cpu().numpy()
    mask_np, _ = compute_foreground_mask(image_np)
    return torch.from_numpy(mask_np).to(image_tensor.device)


def extract_foreground_pieces(dense_features: torch.Tensor, foreground_mask: torch.Tensor) -> torch.Tensor:
    if dense_features.shape[:2] != foreground_mask.shape:
        raise ValueError(
            f"mask shape must match feature grid's spatial shape: "
            f"features={tuple(dense_features.shape[:2])}, mask={tuple(foreground_mask.shape)}"
        )
    return dense_features[foreground_mask]


def cosine_cost_matrix(pieces_a: torch.Tensor, pieces_b: torch.Tensor) -> torch.Tensor:
    a_norm = F.normalize(pieces_a, p=2, dim=1)
    b_norm = F.normalize(pieces_b, p=2, dim=1)
    similarity = a_norm @ b_norm.T
    return 1.0 - similarity


def sinkhorn_ot_distance(cost: torch.Tensor, epsilon: float = 0.05, num_iterations: int = 50) -> torch.Tensor:
    n, m = cost.shape
    log_a = -torch.log(torch.tensor(float(n), device=cost.device, dtype=cost.dtype))
    log_b = -torch.log(torch.tensor(float(m), device=cost.device, dtype=cost.dtype))

    f = torch.zeros(n, device=cost.device, dtype=cost.dtype)
    g = torch.zeros(m, device=cost.device, dtype=cost.dtype)

    for _ in range(num_iterations):
        f = epsilon * log_a - epsilon * torch.logsumexp((g.unsqueeze(0) - cost) / epsilon, dim=1)
        g = epsilon * log_b - epsilon * torch.logsumexp((f.unsqueeze(1) - cost) / epsilon, dim=0)

    transport_plan = torch.exp((f.unsqueeze(1) + g.unsqueeze(0) - cost) / epsilon)
    return (transport_plan * cost).sum()


def dense_matching_distance(
    features_a: torch.Tensor,
    features_b: torch.Tensor,
    mask_a: torch.Tensor,
    mask_b: torch.Tensor,
    epsilon: float = 0.05,
    num_iterations: int = 50,
) -> torch.Tensor:
    pieces_a = extract_foreground_pieces(features_a, mask_a)
    pieces_b = extract_foreground_pieces(features_b, mask_b)

    if pieces_a.shape[0] == 0 or pieces_b.shape[0] == 0:
        raise ValueError(
            f"Cannot compute a matching distance with zero foreground pieces "
            f"(found {pieces_a.shape[0]} in A, {pieces_b.shape[0]} in B)."
        )

    cost = cosine_cost_matrix(pieces_a, pieces_b)
    return sinkhorn_ot_distance(cost, epsilon=epsilon, num_iterations=num_iterations)


# ---------------------------------------------------------------------------
# Step 4: batched Sinkhorn + trainable local projection.
# ---------------------------------------------------------------------------


def _extract_and_pad(dense_features_batch: torch.Tensor, masks_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size, _, _, dim = dense_features_batch.shape
    pieces_list = [dense_features_batch[i][masks_batch[i]] for i in range(batch_size)]
    counts = [p.shape[0] for p in pieces_list]
    if min(counts) == 0:
        raise ValueError(f"Every item needs at least one foreground piece (counts={counts}).")

    max_count = max(counts)
    padded = dense_features_batch.new_zeros(batch_size, max_count, dim)
    valid = torch.zeros(batch_size, max_count, dtype=torch.bool, device=dense_features_batch.device)
    for i, pieces in enumerate(pieces_list):
        n = pieces.shape[0]
        padded[i, :n] = pieces
        valid[i, :n] = True
    return padded, valid


def batched_cosine_cost_matrix(pieces_a: torch.Tensor, pieces_b: torch.Tensor) -> torch.Tensor:
    a_norm = F.normalize(pieces_a, p=2, dim=-1)
    b_norm = F.normalize(pieces_b, p=2, dim=-1)
    similarity = torch.bmm(a_norm, b_norm.transpose(1, 2))
    return 1.0 - similarity


def batched_sinkhorn_ot_distance(
    cost: torch.Tensor,
    valid_a: torch.Tensor,
    valid_b: torch.Tensor,
    epsilon: float = 0.05,
    num_iterations: int = 50,
) -> torch.Tensor:
    batch_size, n, m = cost.shape

    count_a = valid_a.sum(dim=1, keepdim=True).float()
    count_b = valid_b.sum(dim=1, keepdim=True).float()
    if (count_a < 1).any() or (count_b < 1).any():
        raise ValueError("Every item in the batch must have at least one foreground piece.")

    neg_inf = torch.tensor(float("-inf"), device=cost.device, dtype=cost.dtype)
    log_a = torch.where(valid_a, -torch.log(count_a), neg_inf)
    log_b = torch.where(valid_b, -torch.log(count_b), neg_inf)

    f = torch.where(valid_a, torch.zeros_like(log_a), neg_inf)
    g = torch.where(valid_b, torch.zeros_like(log_b), neg_inf)

    for _ in range(num_iterations):
        f = epsilon * log_a - epsilon * torch.logsumexp((g.unsqueeze(1) - cost) / epsilon, dim=2)
        g = epsilon * log_b - epsilon * torch.logsumexp((f.unsqueeze(2) - cost) / epsilon, dim=1)

    transport_plan = torch.exp((f.unsqueeze(2) + g.unsqueeze(1) - cost) / epsilon)
    return (transport_plan * cost).sum(dim=(1, 2))


def batched_dense_matching_distance(
    dense_features_a: torch.Tensor,
    dense_features_b: torch.Tensor,
    masks_a: torch.Tensor,
    masks_b: torch.Tensor,
    local_projection: torch.nn.Module,
    epsilon: float = 0.05,
    num_iterations: int = 50,
) -> torch.Tensor:
    padded_a, valid_a = _extract_and_pad(dense_features_a, masks_a)
    padded_b, valid_b = _extract_and_pad(dense_features_b, masks_b)

    projected_a = local_projection(padded_a)
    projected_b = local_projection(padded_b)

    cost = batched_cosine_cost_matrix(projected_a, projected_b)
    return batched_sinkhorn_ot_distance(cost, valid_a, valid_b, epsilon=epsilon, num_iterations=num_iterations)


def dense_matching_distance_projected(
    features_a: torch.Tensor,
    features_b: torch.Tensor,
    mask_a: torch.Tensor,
    mask_b: torch.Tensor,
    local_projection: torch.nn.Module,
    epsilon: float = 0.05,
    num_iterations: int = 50,
) -> torch.Tensor:
    distances = batched_dense_matching_distance(
        features_a.unsqueeze(0), features_b.unsqueeze(0),
        mask_a.unsqueeze(0), mask_b.unsqueeze(0),
        local_projection, epsilon=epsilon, num_iterations=num_iterations,
    )
    return distances.squeeze(0)


In [ ]:
%%writefile /kaggle/working/supervised_src/combined_distance.py
"""Step 4: the combined distance - global (Method A) + structural (Method
B) - Kaggle version (flat imports). Per DetailSemNet's Eq. 2:

    dis = lambda_0 * dis_global + dis_struct
"""

from __future__ import annotations

import torch
import torch.nn as nn

from mask import compute_foreground_mask
from dense_matching import batched_dense_matching_distance


def compute_masks_from_tensor_batch(image_batch: torch.Tensor) -> torch.Tensor:
    masks = []
    for image in image_batch:
        image_np = image.squeeze().detach().cpu().numpy()
        mask_np, _ = compute_foreground_mask(image_np)
        masks.append(torch.from_numpy(mask_np))
    return torch.stack(masks).to(image_batch.device)


def combined_distance_batched(
    model: nn.Module,
    images_a: torch.Tensor,
    images_b: torch.Tensor,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> torch.Tensor:
    """`(B, 1, H, W)` x2 -> `(B,)` combined distances, fully differentiable
    through the global projector, `model.local_projection`, and the
    encoder (whichever stages are unfrozen)."""
    if model.local_projection is None:
        raise ValueError(
            "model.local_projection is None - build the model with "
            "local_embedding_dim set to use the combined distance."
        )

    global_a = model.forward_global(images_a)
    global_b = model.forward_global(images_b)
    dis_global = torch.norm(global_a - global_b, p=2, dim=1)

    dense_a = model.forward_dense(images_a)
    dense_b = model.forward_dense(images_b)
    masks_a = compute_masks_from_tensor_batch(images_a)
    masks_b = compute_masks_from_tensor_batch(images_b)

    dis_struct = batched_dense_matching_distance(
        dense_a, dense_b, masks_a, masks_b, model.local_projection,
        epsilon=sinkhorn_epsilon, num_iterations=sinkhorn_iterations,
    )

    return lambda_0 * dis_global + dis_struct


In [ ]:
%%writefile /kaggle/working/supervised_src/run_one_epoch.py
"""Runs one training or evaluation pass over a dual-triplet dataloader.

Adapted from the SURDS-era `run_one_epoch.py`
(`Thesis_Final/downstream_verification/utils/run_one_epoch.py`) - same
per-batch mechanics (compute the 4 embeddings via the model's default
forward, compute the dual-triplet loss, track ranking-accuracy
diagnostics, backward + optimizer step only when an optimizer is given).
Unchanged from the old pipeline: `DownstreamVerificationModel.forward()`
already defaults to `forward_global` (Method A), so `model(x)` here is
Method A exactly, same as the old model's single forward path.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm


def _compute_batch_embeddings(model: nn.Module, batch: dict, device: torch.device) -> dict[str, torch.Tensor]:
    anchor = batch["anchor"].to(device)
    positive = batch["positive"].to(device)
    negative_intra = batch["negative_intra"].to(device)
    negative_inter = batch["negative_inter"].to(device)

    return {
        "anchor_embedding": model(anchor),
        "positive_embedding": model(positive),
        "negative_intra_embedding": model(negative_intra),
        "negative_inter_embedding": model(negative_inter),
    }


def _compute_distance_diagnostics(
    anchor_embedding: torch.Tensor,
    positive_embedding: torch.Tensor,
    negative_intra_embedding: torch.Tensor,
    negative_inter_embedding: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """Beyond the loss number itself: are genuine pairs actually ending up
    closer than forgeries/other-writers, on average, in this batch? A
    falling loss doesn't guarantee this - these ranking-accuracy numbers
    are a more direct per-epoch read on whether embeddings are ordering
    correctly, independent of the loss margin's exact scale."""
    positive_distance = F.pairwise_distance(anchor_embedding, positive_embedding)
    negative_intra_distance = F.pairwise_distance(anchor_embedding, negative_intra_embedding)
    negative_inter_distance = F.pairwise_distance(anchor_embedding, negative_inter_embedding)

    return {
        "positive_distance_mean": positive_distance.mean(),
        "negative_intra_distance_mean": negative_intra_distance.mean(),
        "negative_inter_distance_mean": negative_inter_distance.mean(),
        "intra_ranking_accuracy": (positive_distance < negative_intra_distance).float().mean(),
        "inter_ranking_accuracy": (positive_distance < negative_inter_distance).float().mean(),
    }


def run_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    description: str = "Eval",
) -> dict[str, float]:
    """One pass over `data_loader`. Training mode (backward + optimizer
    step) iff `optimizer` is given; otherwise a forward-only evaluation
    pass - used for the fixed-quadruple validation loss (see
    `downstream_supervised_learning_approach.md`, "Validation loss curve:
    fixed sampling, not dynamic" - a training-health diagnostic, not the
    real checkpoint-selection metric)."""
    is_training = optimizer is not None
    model.train(mode=is_training)

    running = {
        "loss": 0.0, "intra_loss": 0.0, "inter_loss": 0.0,
        "positive_distance_mean": 0.0, "negative_intra_distance_mean": 0.0, "negative_inter_distance_mean": 0.0,
        "intra_ranking_accuracy": 0.0, "inter_ranking_accuracy": 0.0,
    }
    num_batches = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        progress_bar = tqdm(data_loader, desc=description, ncols=100, mininterval=10)
        for batch in progress_bar:
            if is_training:
                optimizer.zero_grad()

            embeddings = _compute_batch_embeddings(model, batch, device)
            loss_outputs = loss_function(**embeddings)
            diagnostics = _compute_distance_diagnostics(**embeddings)

            if is_training:
                loss_outputs["loss"].backward()
                optimizer.step()

            for key in ("loss", "intra_loss", "inter_loss"):
                running[key] += float(loss_outputs[key].item())
            for key in (
                "positive_distance_mean", "negative_intra_distance_mean", "negative_inter_distance_mean",
                "intra_ranking_accuracy", "inter_ranking_accuracy",
            ):
                running[key] += float(diagnostics[key].item())
            num_batches += 1

            progress_bar.set_postfix({
                "loss": f"{running['loss'] / num_batches:.4f}",
                "intra_acc": f"{running['intra_ranking_accuracy'] / num_batches:.4f}",
                "inter_acc": f"{running['inter_ranking_accuracy'] / num_batches:.4f}",
            })

    return {key: value / max(1, num_batches) for key, value in running.items()}


In [ ]:
%%writefile /kaggle/working/supervised_src/run_one_epoch_combined.py
"""Runs one training or evaluation pass using Step 4's combined distance
(global + local/structural, both branches trainable via a shared encoder
forward pass) - Kaggle version (flat imports).
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from combined_distance import combined_distance_batched

_METRIC_KEYS = (
    "loss", "positive_loss", "negative_loss",
    "positive_distance_mean", "negative_distance_mean",
    "positive_active_rate", "negative_active_rate",
)


def run_one_epoch_combined(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
    description: str = "Eval",
) -> dict[str, float]:
    is_training = optimizer is not None
    model.train(mode=is_training)

    running = {key: 0.0 for key in _METRIC_KEYS}
    num_batches = 0

    context_manager = torch.enable_grad() if is_training else torch.no_grad()
    with context_manager:
        progress_bar = tqdm(data_loader, desc=description, ncols=100, mininterval=10)
        for batch in progress_bar:
            if is_training:
                optimizer.zero_grad()

            image_a = batch["image_a"].to(device)
            image_b = batch["image_b"].to(device)
            label = batch["label"].to(device).float()

            dist = combined_distance_batched(
                model, image_a, image_b,
                lambda_0=lambda_0, sinkhorn_epsilon=sinkhorn_epsilon, sinkhorn_iterations=sinkhorn_iterations,
            )
            loss_outputs = loss_function(dist, label)

            if is_training:
                loss_outputs["loss"].backward()
                optimizer.step()

            for key in _METRIC_KEYS:
                running[key] += float(loss_outputs[key].item())
            num_batches += 1

            progress_bar.set_postfix({
                "loss": f"{running['loss'] / num_batches:.4f}",
                "pos_active": f"{running['positive_active_rate'] / num_batches:.4f}",
                "neg_active": f"{running['negative_active_rate'] / num_batches:.4f}",
            })

    return {key: value / max(1, num_batches) for key, value in running.items()}


In [ ]:
%%writefile /kaggle/working/supervised_src/train_validation.py
"""Epoch loop for downstream supervised verification training - Kaggle
version (flat imports). Logic identical to the local project's
train_validation.py - see that module's docstring for the full design
rationale (verification_callback as the real checkpoint-selection metric,
run_epoch_fn as the loss-path extension point, per-epoch CSV logging).
"""

from __future__ import annotations

from pathlib import Path
from typing import Callable, Optional

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from run_one_epoch import run_one_epoch


def train_and_validate_model(
    model: nn.Module,
    train_dataset,
    train_loader: DataLoader,
    val_loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    device: torch.device,
    history_csv_path: Path,
    model_dir: Path,
    best_model_name: str = "best_model.pt",
    periodic_save_frequency: int = 5,
    verification_callback: Optional[Callable[[nn.Module, torch.device], dict]] = None,
    run_epoch_fn: Optional[Callable] = None,
    patience: Optional[int] = None,
    min_epochs: Optional[int] = None,
) -> pd.DataFrame:
    run_epoch_fn = run_epoch_fn or run_one_epoch
    if verification_callback is None:
        print(
            "WARNING: no verification_callback given - checkpoint selection is "
            "falling back to VALIDATION TRIPLET LOSS (a documented placeholder, "
            "not the intended methodology - pass verification_callback for the "
            "real K-reference AUC/EER protocol)."
        )

    history_csv_path.parent.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    history: list[dict] = []
    best_score = float("-inf") if verification_callback is not None else float("inf")
    epochs_since_best = 0

    for epoch in range(epochs):
        current_epoch = epoch + 1
        print(f"===== Epoch {current_epoch}/{epochs} =====")

        train_dataset.set_epoch(epoch)

        train_metrics = run_epoch_fn(
            model=model, data_loader=train_loader, loss_function=loss_function,
            device=device, optimizer=optimizer, description="Train",
        )
        val_metrics = run_epoch_fn(
            model=model, data_loader=val_loader, loss_function=loss_function,
            device=device, optimizer=None, description="Val loss (fixed quadruples)",
        )

        epoch_record: dict = {"epoch": current_epoch}
        for key, value in train_metrics.items():
            epoch_record[f"train_{key}"] = value
        for key, value in val_metrics.items():
            epoch_record[f"val_{key}"] = value

        if verification_callback is not None:
            print("Running verification protocol...")
            verification_metrics = verification_callback(model, device)
            epoch_record.update(verification_metrics)
            current_score = verification_metrics["primary_metric"]
            improved = current_score > best_score
        else:
            current_score = val_metrics["loss"]
            improved = current_score < best_score

        history.append(epoch_record)
        print(epoch_record)

        if improved:
            best_score = current_score
            epochs_since_best = 0
            best_model_path = model_dir / best_model_name
            torch.save({
                "epoch": current_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_score": best_score,
                "history": history,
            }, best_model_path)
            print(f"Best model saved -> {best_model_path}")
        else:
            epochs_since_best += 1

        if current_epoch % periodic_save_frequency == 0:
            periodic_path = model_dir / f"epoch{current_epoch}.pt"
            torch.save({
                "epoch": current_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_metrics["loss"],
                "history": history,
            }, periodic_path)
            print(f"Periodic checkpoint saved -> {periodic_path}")

        history_df = pd.DataFrame(history)
        history_df.to_csv(history_csv_path, index=False)

        if patience is not None and current_epoch >= (min_epochs or epochs) and epochs_since_best >= patience:
            print(
                f"Early stopping at epoch {current_epoch} "
                f"(no improvement in {patience} epochs; best={best_score:.4f})"
            )
            break

    return pd.DataFrame(history)


In [ ]:
%%writefile /kaggle/working/supervised_src/encoding.py
"""Step 5, part 1: cache every signature's representation once - Kaggle
version (flat imports). Logic identical to the local project's
evaluation/encoding.py.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm

from preprocess import preprocess_signature
from mask import compute_foreground_mask
from embedding_model import DownstreamVerificationModel
from dense_matching import (
    batched_dense_matching_distance,
    dense_matching_distance,
    dense_matching_distance_projected,
)


@dataclass(frozen=True)
class EncodedSignature:
    global_embedding: torch.Tensor  # (embedding_dim,) - Method A
    dense_features: torch.Tensor    # (grid, grid, feature_dim) - Method B
    foreground_mask: torch.Tensor   # (grid, grid) bool - Method B


@torch.no_grad()
def encode_signature(model: DownstreamVerificationModel, image_path: Path, device: torch.device) -> EncodedSignature:
    raw = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if raw is None:
        raise FileNotFoundError(f"Could not read image at {image_path}")
    view = preprocess_signature(raw)
    mask_np, _ = compute_foreground_mask(view)

    tensor = torch.from_numpy((view > 0).astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    global_embedding = model.forward_global(tensor).squeeze(0)
    dense_features = model.forward_dense(tensor).squeeze(0)

    return EncodedSignature(
        global_embedding=global_embedding,
        dense_features=dense_features,
        foreground_mask=torch.from_numpy(mask_np).to(device),
    )


def encode_all_signatures(
    model: DownstreamVerificationModel,
    image_paths: list[Path],
    device: torch.device,
    show_progress: bool = True,
) -> dict[Path, EncodedSignature]:
    model.eval()
    unique_paths = sorted(set(image_paths))
    iterator = tqdm(unique_paths, desc="Encoding signatures", ncols=100) if show_progress else unique_paths
    return {path: encode_signature(model, path, device) for path in iterator}


def method_a_distance_cached(encoded_a: EncodedSignature, encoded_b: EncodedSignature) -> float:
    return torch.norm(encoded_a.global_embedding - encoded_b.global_embedding, p=2).item()


def method_b_distance_cached(
    encoded_a: EncodedSignature,
    encoded_b: EncodedSignature,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> float:
    return dense_matching_distance(
        encoded_a.dense_features, encoded_b.dense_features,
        encoded_a.foreground_mask, encoded_b.foreground_mask,
        epsilon=sinkhorn_epsilon, num_iterations=sinkhorn_iterations,
    ).item()


def blended_distance_cached(
    encoded_a: EncodedSignature,
    encoded_b: EncodedSignature,
    alpha: float,
    method_a_scale: float,
    method_b_scale: float,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> float:
    if not 0.0 <= alpha <= 1.0:
        raise ValueError(f"alpha must be in [0, 1], got {alpha}")

    component_a = 0.0
    component_b = 0.0
    if alpha < 1.0:
        component_a = (1.0 - alpha) * (method_a_distance_cached(encoded_a, encoded_b) / method_a_scale)
    if alpha > 0.0:
        component_b = alpha * (
            method_b_distance_cached(encoded_a, encoded_b, sinkhorn_epsilon, sinkhorn_iterations) / method_b_scale
        )
    return component_a + component_b


def combined_distance_cached(
    encoded_a: EncodedSignature,
    encoded_b: EncodedSignature,
    local_projection: torch.nn.Module,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> float:
    dis_global = method_a_distance_cached(encoded_a, encoded_b)
    dis_struct = dense_matching_distance_projected(
        encoded_a.dense_features, encoded_b.dense_features,
        encoded_a.foreground_mask, encoded_b.foreground_mask,
        local_projection, epsilon=sinkhorn_epsilon, num_iterations=sinkhorn_iterations,
    ).item()
    return lambda_0 * dis_global + dis_struct


def combined_distance_batch_cached(
    pairs: list[tuple[EncodedSignature, EncodedSignature]],
    local_projection: torch.nn.Module,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> list[float]:
    if not pairs:
        return []

    with torch.no_grad():
        embeddings_a = torch.stack([a.global_embedding for a, _ in pairs])
        embeddings_b = torch.stack([b.global_embedding for _, b in pairs])
        dis_global = torch.norm(embeddings_a - embeddings_b, p=2, dim=1)

        dense_a = torch.stack([a.dense_features for a, _ in pairs])
        dense_b = torch.stack([b.dense_features for _, b in pairs])
        masks_a = torch.stack([a.foreground_mask for a, _ in pairs])
        masks_b = torch.stack([b.foreground_mask for _, b in pairs])

        dis_struct = batched_dense_matching_distance(
            dense_a, dense_b, masks_a, masks_b, local_projection,
            epsilon=sinkhorn_epsilon, num_iterations=sinkhorn_iterations,
        )
        dis_combined = lambda_0 * dis_global + dis_struct

    return dis_combined.tolist()


In [ ]:
%%writefile /kaggle/working/supervised_src/scoring.py
"""Step 5, part 2: reference/query splitting and per-reference-then-average
scoring - Kaggle version (flat imports). Logic identical to the local
project's evaluation/scoring.py.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
from tqdm import tqdm

from dual_triplet_dataset import list_genuine_and_forged_images
from embedding_model import DownstreamVerificationModel
from encoding import EncodedSignature, encode_all_signatures

DistanceFn = Callable[[EncodedSignature, EncodedSignature], float]
BatchDistanceFn = Callable[[list[tuple[EncodedSignature, EncodedSignature]]], list[float]]


@dataclass(frozen=True)
class QueryScore:
    writer_id: str
    query_path: Path
    query_type: str
    label: int
    distance: float
    score: float


@dataclass(frozen=True)
class WriterSplit:
    reference_paths: list[Path]
    genuine_query_paths: list[Path]
    forgery_query_paths: list[Path]


def split_references_and_queries(
    writer_dir: Path,
    dataset_name: str,
    num_references: int,
    seed: int,
) -> WriterSplit:
    genuine_paths, forgery_paths = list_genuine_and_forged_images(writer_dir, dataset_name)
    if len(genuine_paths) <= num_references:
        raise ValueError(
            f"Writer {writer_dir.name} has only {len(genuine_paths)} genuine signatures; "
            f"need more than {num_references} for a reference + query split."
        )
    if not forgery_paths:
        raise ValueError(f"Writer {writer_dir.name} has no forgery signatures.")

    rng = random.Random(seed)
    reference_paths = sorted(rng.sample(genuine_paths, num_references))
    reference_set = set(reference_paths)
    genuine_query_paths = [p for p in genuine_paths if p not in reference_set]

    return WriterSplit(
        reference_paths=reference_paths,
        genuine_query_paths=genuine_query_paths,
        forgery_query_paths=forgery_paths,
    )


def score_writer(
    writer_id: str,
    split: WriterSplit,
    encoded: dict[Path, EncodedSignature],
    distance_fn: DistanceFn,
) -> list[QueryScore]:
    scores: list[QueryScore] = []
    labeled_queries = (
        [(p, "genuine", 1) for p in split.genuine_query_paths]
        + [(p, "forgery", 0) for p in split.forgery_query_paths]
    )

    for query_path, query_type, label in labeled_queries:
        per_reference_distances = [
            distance_fn(encoded[query_path], encoded[reference_path])
            for reference_path in split.reference_paths
        ]
        distance = float(np.mean(per_reference_distances))
        scores.append(QueryScore(
            writer_id=writer_id, query_path=query_path, query_type=query_type,
            label=label, distance=distance, score=-distance,
        ))

    return scores


def _score_all_writers_batched(
    writer_splits: dict[str, WriterSplit],
    encoded: dict[Path, EncodedSignature],
    batch_distance_fn: BatchDistanceFn,
    batch_size: int,
    show_progress: bool,
) -> list[QueryScore]:
    rows: list[tuple[str, Path, str, int, Path]] = []
    for writer_id, split in writer_splits.items():
        labeled_queries = (
            [(p, "genuine", 1) for p in split.genuine_query_paths]
            + [(p, "forgery", 0) for p in split.forgery_query_paths]
        )
        for query_path, query_type, label in labeled_queries:
            for reference_path in split.reference_paths:
                rows.append((writer_id, query_path, query_type, label, reference_path))

    if not rows:
        return []

    num_chunks = (len(rows) + batch_size - 1) // batch_size
    chunk_starts = range(0, len(rows), batch_size)
    if show_progress:
        chunk_starts = tqdm(chunk_starts, desc="Scoring pairs (batched)", ncols=100, total=num_chunks)

    all_distances: list[float] = []
    for start in chunk_starts:
        chunk = rows[start : start + batch_size]
        pairs = [(encoded[query_path], encoded[reference_path]) for _, query_path, _, _, reference_path in chunk]
        all_distances.extend(batch_distance_fn(pairs))

    if len(all_distances) != len(rows):
        raise ValueError(
            f"batch_distance_fn returned {len(all_distances)} distances for {len(rows)} pairs - "
            "it must return exactly one distance per input pair, in order."
        )

    per_query_distances: dict[tuple[str, Path], list[float]] = {}
    per_query_meta: dict[tuple[str, Path], tuple[str, int]] = {}
    for (writer_id, query_path, query_type, label, _reference_path), distance in zip(rows, all_distances):
        key = (writer_id, query_path)
        per_query_distances.setdefault(key, []).append(distance)
        per_query_meta[key] = (query_type, label)

    scores: list[QueryScore] = []
    for (writer_id, query_path), distances in per_query_distances.items():
        query_type, label = per_query_meta[(writer_id, query_path)]
        mean_distance = float(np.mean(distances))
        scores.append(QueryScore(
            writer_id=writer_id, query_path=query_path, query_type=query_type,
            label=label, distance=mean_distance, score=-mean_distance,
        ))

    return scores


def run_verification_draw(
    model: DownstreamVerificationModel,
    writer_ids: list[str],
    dataset_dir: Path,
    dataset_name: str,
    distance_fn: DistanceFn,
    num_references: int,
    seed: int,
    device: torch.device,
    show_progress: bool = True,
    encoded_cache: Optional[dict[Path, EncodedSignature]] = None,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> list[QueryScore]:
    writer_splits: dict[str, WriterSplit] = {}
    all_paths: list[Path] = []

    for writer_id in writer_ids:
        writer_dir = dataset_dir / writer_id
        split = split_references_and_queries(writer_dir, dataset_name, num_references, seed)
        writer_splits[writer_id] = split
        all_paths.extend(split.reference_paths)
        all_paths.extend(split.genuine_query_paths)
        all_paths.extend(split.forgery_query_paths)

    if encoded_cache is None:
        encoded = encode_all_signatures(model, all_paths, device, show_progress=show_progress)
    else:
        missing = [p for p in dict.fromkeys(all_paths) if p not in encoded_cache]
        if missing:
            encoded_cache.update(
                encode_all_signatures(model, missing, device, show_progress=show_progress)
            )
        encoded = encoded_cache

    if batch_distance_fn is not None:
        return _score_all_writers_batched(writer_splits, encoded, batch_distance_fn, batch_size, show_progress)

    all_scores: list[QueryScore] = []
    for writer_id, split in writer_splits.items():
        all_scores.extend(score_writer(writer_id, split, encoded, distance_fn))

    return all_scores


In [ ]:
%%writefile /kaggle/working/supervised_src/metrics.py
"""Step 5, part 3: threshold-free metrics (ROC-AUC, EER) from a scored draw.

Direct port of the SURDS-era metrics math
(`Thesis_Final/downstream_verification/evaluation/metrics.py`), adapted to
operate on `list[QueryScore]` (from `scoring.py`) instead of a pandas
DataFrame.
"""

from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from sklearn.metrics import auc, roc_curve

EVALUATION_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(EVALUATION_DIR))

from scoring import QueryScore  # noqa: E402


@dataclass(frozen=True)
class RocResult:
    fpr: np.ndarray
    tpr: np.ndarray
    thresholds: np.ndarray  # on the SCORE scale (-distance), matching sklearn's roc_curve convention
    roc_auc: float


def compute_roc_metrics(scores: list[QueryScore]) -> RocResult:
    labels = np.array([s.label for s in scores])
    score_values = np.array([s.score for s in scores])
    fpr, tpr, thresholds = roc_curve(labels, score_values)
    roc_auc = float(auc(fpr, tpr))
    return RocResult(fpr=fpr, tpr=tpr, thresholds=thresholds, roc_auc=roc_auc)


def compute_eer(roc: RocResult) -> dict[str, float]:
    """EER at the point where FPR is closest to FNR (1 - TPR)."""
    fnr = 1.0 - roc.tpr
    eer_gap = np.abs(roc.fpr - fnr)
    index = int(np.argmin(eer_gap))
    return {
        "eer": float((roc.fpr[index] + fnr[index]) / 2.0),
        "eer_score_threshold": float(roc.thresholds[index]),
        "eer_distance_threshold": float(-roc.thresholds[index]),
    }


def compute_per_writer_auc(scores: list[QueryScore]) -> dict[str, float]:
    """AUC computed independently per writer. Writers with only one label
    class present (no forgery queries, or no genuine queries left after
    the reference split) are skipped - AUC is undefined for a single
    class."""
    writer_ids = sorted({s.writer_id for s in scores})
    per_writer_auc: dict[str, float] = {}

    for writer_id in writer_ids:
        writer_scores = [s for s in scores if s.writer_id == writer_id]
        labels = {s.label for s in writer_scores}
        if len(labels) < 2:
            print(f"Warning: writer {writer_id} has only one label class - skipping per-writer AUC.")
            continue
        roc = compute_roc_metrics(writer_scores)
        per_writer_auc[writer_id] = roc.roc_auc

    return per_writer_auc


In [ ]:
%%writefile /kaggle/working/supervised_src/threshold.py
"""Step 5, part 4: leakage-free threshold (tau) selection and application -
Kaggle version (flat imports). Logic identical to the local project's
evaluation/threshold.py. Not exercised by this notebook's training run
itself (build_verification_callback never sweeps tau - see protocol.py),
but protocol.py imports this module unconditionally, so it must be present.
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional

import numpy as np
import torch

from embedding_model import DownstreamVerificationModel
from metrics import compute_eer, compute_roc_metrics
from scoring import BatchDistanceFn, DistanceFn, QueryScore, run_verification_draw

DEFAULT_TAU_STEP = 0.00005


def metrics_at_threshold(distances: np.ndarray, labels: np.ndarray, tau: float) -> dict[str, float]:
    distances = np.asarray(distances, dtype=np.float64)
    labels = np.asarray(labels).astype(int)

    predicted_genuine = distances <= tau
    is_genuine = labels == 1
    is_forgery = labels == 0

    true_positive = int(np.sum(predicted_genuine & is_genuine))
    false_negative = int(np.sum(~predicted_genuine & is_genuine))
    true_negative = int(np.sum(~predicted_genuine & is_forgery))
    false_positive = int(np.sum(predicted_genuine & is_forgery))
    total = true_positive + false_negative + true_negative + false_positive

    num_genuine = true_positive + false_negative
    num_forgery = true_negative + false_positive

    tpr = true_positive / num_genuine if num_genuine > 0 else 0.0
    tnr = true_negative / num_forgery if num_forgery > 0 else 0.0
    fnr = 1.0 - tpr
    fpr = 1.0 - tnr

    return {
        "tau": float(tau),
        "tp": true_positive, "fn": false_negative, "tn": true_negative, "fp": false_positive,
        "accuracy": float((true_positive + true_negative) / total) if total > 0 else 0.0,
        "balanced_accuracy": float((tpr + tnr) / 2.0),
        "tpr": float(tpr), "tnr": float(tnr), "fpr": float(fpr), "fnr": float(fnr),
        "far": float(fpr), "frr": float(fnr),
    }


def sweep_balanced_accuracy_threshold(
    distances: np.ndarray, labels: np.ndarray, step: float = DEFAULT_TAU_STEP,
) -> dict[str, float]:
    distances = np.asarray(distances, dtype=np.float64)
    labels = np.asarray(labels).astype(int)
    if distances.size == 0:
        raise ValueError("Cannot sweep tau over an empty distance array.")

    lowest, highest = float(distances.min()), float(distances.max())
    candidate_taus = np.arange(lowest, highest + step, step)

    genuine_sorted = np.sort(distances[labels == 1])
    forgery_sorted = np.sort(distances[labels == 0])
    num_genuine, num_forgery = genuine_sorted.size, forgery_sorted.size

    true_positive = np.searchsorted(genuine_sorted, candidate_taus, side="right")
    false_positive = np.searchsorted(forgery_sorted, candidate_taus, side="right")
    true_negative = num_forgery - false_positive

    tpr = true_positive / num_genuine if num_genuine > 0 else np.zeros_like(candidate_taus)
    tnr = true_negative / num_forgery if num_forgery > 0 else np.zeros_like(candidate_taus)
    balanced_accuracy = (tpr + tnr) / 2.0

    best_tau = float(candidate_taus[int(np.argmax(balanced_accuracy))])
    return metrics_at_threshold(distances, labels, best_tau)


def _scores_to_arrays(scores: list[QueryScore]) -> tuple[np.ndarray, np.ndarray]:
    distances = np.array([s.distance for s in scores], dtype=np.float64)
    labels = np.array([s.label for s in scores], dtype=int)
    return distances, labels


def select_tau_per_draw_average(
    model: DownstreamVerificationModel,
    writer_ids: list[str],
    dataset_dir: Path,
    dataset_name: str,
    distance_fn: DistanceFn,
    num_references: int,
    seeds: list[int],
    device: torch.device,
    step: float = DEFAULT_TAU_STEP,
    show_progress: bool = True,
    encoded_cache: Optional[dict] = None,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> dict[str, float]:
    per_draw_taus: list[float] = []
    per_draw_balanced_accs: list[float] = []

    for seed in seeds:
        scores = run_verification_draw(
            model, writer_ids, dataset_dir, dataset_name, distance_fn,
            num_references, seed, device, show_progress=show_progress,
            encoded_cache=encoded_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
        )
        distances, labels = _scores_to_arrays(scores)
        best = sweep_balanced_accuracy_threshold(distances, labels, step=step)
        per_draw_taus.append(best["tau"])
        per_draw_balanced_accs.append(best["balanced_accuracy"])

    return {
        "tau_star": float(np.mean(per_draw_taus)),
        "tau_std": float(np.std(per_draw_taus)),
        "val_balanced_acc_mean": float(np.mean(per_draw_balanced_accs)),
        "val_balanced_acc_std": float(np.std(per_draw_balanced_accs)),
        "per_draw_taus": per_draw_taus,
    }


def evaluate_test_optimal_tau_per_draw(
    model: DownstreamVerificationModel,
    writer_ids: list[str],
    dataset_dir: Path,
    dataset_name: str,
    distance_fn: DistanceFn,
    num_references: int,
    seeds: list[int],
    device: torch.device,
    step: float = DEFAULT_TAU_STEP,
    show_progress: bool = True,
    encoded_cache: Optional[dict] = None,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> dict[str, float]:
    accumulators: dict[str, list[float]] = {
        "roc_auc": [], "eer": [], "tau": [], "accuracy": [], "balanced_accuracy": [],
        "tpr": [], "tnr": [], "far": [], "frr": [],
    }

    for seed in seeds:
        scores = run_verification_draw(
            model, writer_ids, dataset_dir, dataset_name, distance_fn,
            num_references, seed, device, show_progress=show_progress,
            encoded_cache=encoded_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
        )
        distances, labels = _scores_to_arrays(scores)

        roc = compute_roc_metrics(scores)
        best = sweep_balanced_accuracy_threshold(distances, labels, step=step)

        accumulators["roc_auc"].append(roc.roc_auc)
        accumulators["eer"].append(compute_eer(roc)["eer"])
        for key in ("tau", "accuracy", "balanced_accuracy", "tpr", "tnr", "far", "frr"):
            accumulators[key].append(best[key])

    summary: dict[str, float] = {}
    for metric_name, values in accumulators.items():
        summary[f"{metric_name}_mean"] = float(np.mean(values))
        summary[f"{metric_name}_std"] = float(np.std(values))
    return summary


def evaluate_at_fixed_tau_per_draw(
    model: DownstreamVerificationModel,
    writer_ids: list[str],
    dataset_dir: Path,
    dataset_name: str,
    distance_fn: DistanceFn,
    num_references: int,
    seeds: list[int],
    tau: float,
    device: torch.device,
    show_progress: bool = True,
    encoded_cache: Optional[dict] = None,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> dict[str, float]:
    accumulators: dict[str, list[float]] = {
        "roc_auc": [], "eer": [], "accuracy": [], "balanced_accuracy": [],
        "tpr": [], "tnr": [], "far": [], "frr": [],
    }

    for seed in seeds:
        scores = run_verification_draw(
            model, writer_ids, dataset_dir, dataset_name, distance_fn,
            num_references, seed, device, show_progress=show_progress,
            encoded_cache=encoded_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
        )
        distances, labels = _scores_to_arrays(scores)

        roc = compute_roc_metrics(scores)
        eer_info = compute_eer(roc)
        threshold_metrics = metrics_at_threshold(distances, labels, tau)

        accumulators["roc_auc"].append(roc.roc_auc)
        accumulators["eer"].append(eer_info["eer"])
        for key in ("accuracy", "balanced_accuracy", "tpr", "tnr", "far", "frr"):
            accumulators[key].append(threshold_metrics[key])

    summary: dict[str, float] = {"tau": float(tau)}
    for metric_name, values in accumulators.items():
        summary[f"{metric_name}_mean"] = float(np.mean(values))
        summary[f"{metric_name}_std"] = float(np.std(values))
    return summary


In [ ]:
%%writefile /kaggle/working/supervised_src/protocol.py
"""Step 5, part 5: top-level orchestration - Kaggle version (flat imports).
Logic identical to the local project's evaluation/protocol.py. Only
`build_verification_callback` + `make_combined_distance_fn` +
`make_combined_batch_distance_fn` are exercised by this notebook's
training run (per-epoch checkpoint selection); `run_full_protocol` is kept
for completeness (a future test-set evaluation notebook/cell would use it)
but is not called here.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn

from encoding import (
    blended_distance_cached,
    combined_distance_batch_cached,
    combined_distance_cached,
    method_a_distance_cached,
    method_b_distance_cached,
)
from metrics import compute_eer, compute_per_writer_auc, compute_roc_metrics
from scoring import BatchDistanceFn, DistanceFn, run_verification_draw
from threshold import (
    evaluate_at_fixed_tau_per_draw,
    evaluate_test_optimal_tau_per_draw,
    select_tau_per_draw_average,
)


def make_method_a_distance_fn() -> DistanceFn:
    return method_a_distance_cached


def make_method_b_distance_fn(sinkhorn_epsilon: float = 0.05, sinkhorn_iterations: int = 50) -> DistanceFn:
    return lambda a, b: method_b_distance_cached(a, b, sinkhorn_epsilon, sinkhorn_iterations)


def make_blended_distance_fn(
    alpha: float,
    method_a_scale: float,
    method_b_scale: float,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> DistanceFn:
    return lambda a, b: blended_distance_cached(
        a, b, alpha, method_a_scale, method_b_scale, sinkhorn_epsilon, sinkhorn_iterations,
    )


def make_combined_distance_fn(
    model: nn.Module,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> DistanceFn:
    return lambda a, b: combined_distance_cached(
        a, b, model.local_projection, lambda_0, sinkhorn_epsilon, sinkhorn_iterations,
    )


def make_combined_batch_distance_fn(
    model: nn.Module,
    lambda_0: float = 1.0,
    sinkhorn_epsilon: float = 0.05,
    sinkhorn_iterations: int = 50,
) -> BatchDistanceFn:
    return lambda pairs: combined_distance_batch_cached(
        pairs, model.local_projection, lambda_0, sinkhorn_epsilon, sinkhorn_iterations,
    )


def build_verification_callback(
    dataset_dir: Path,
    dataset_name: str,
    validation_writer_ids: list[str],
    distance_fn: DistanceFn,
    num_references: int = 5,
    seeds: tuple[int, ...] = (101, 202),
    show_progress: bool = False,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> Callable[[nn.Module, torch.device], dict]:
    def callback(model: nn.Module, device: torch.device) -> dict:
        encoded_cache: dict = {}
        aucs: list[float] = []
        eers: list[float] = []
        for seed in seeds:
            scores = run_verification_draw(
                model, validation_writer_ids, dataset_dir, dataset_name, distance_fn,
                num_references, seed, device, show_progress=show_progress,
                encoded_cache=encoded_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
            )
            roc = compute_roc_metrics(scores)
            aucs.append(roc.roc_auc)
            eers.append(compute_eer(roc)["eer"])

        mean_auc = float(np.mean(aucs))
        return {
            "primary_metric": mean_auc,
            "val_proto_auc_mean": mean_auc,
            "val_proto_auc_std": float(np.std(aucs)),
            "val_proto_eer_mean": float(np.mean(eers)),
        }

    return callback


@dataclass(frozen=True)
class ProtocolResult:
    val_tau_star: float
    val_balanced_acc_mean: float
    test_summary: dict[str, float]
    test_summary_surds_convention: dict[str, float]
    per_writer_auc: dict[str, float]
    num_references: int


def run_full_protocol(
    model: nn.Module,
    dataset_dir: Path,
    dataset_name: str,
    distance_fn: DistanceFn,
    validation_writer_ids: list[str],
    test_writer_ids: list[str],
    num_references: int,
    val_seeds: list[int],
    test_seeds: list[int],
    device: torch.device,
    show_progress: bool = True,
    batch_distance_fn: Optional[BatchDistanceFn] = None,
    batch_size: int = 64,
) -> ProtocolResult:
    val_cache: dict = {}
    test_cache: dict = {}

    tau_selection = select_tau_per_draw_average(
        model, validation_writer_ids, dataset_dir, dataset_name, distance_fn,
        num_references, val_seeds, device, show_progress=show_progress,
        encoded_cache=val_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
    )
    test_summary = evaluate_at_fixed_tau_per_draw(
        model, test_writer_ids, dataset_dir, dataset_name, distance_fn,
        num_references, test_seeds, tau_selection["tau_star"], device, show_progress=show_progress,
        encoded_cache=test_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
    )
    test_summary_surds = evaluate_test_optimal_tau_per_draw(
        model, test_writer_ids, dataset_dir, dataset_name, distance_fn,
        num_references, test_seeds, device, show_progress=show_progress,
        encoded_cache=test_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
    )
    diagnostic_scores = run_verification_draw(
        model, test_writer_ids, dataset_dir, dataset_name, distance_fn,
        num_references, test_seeds[-1], device, show_progress=False,
        encoded_cache=test_cache, batch_distance_fn=batch_distance_fn, batch_size=batch_size,
    )
    per_writer_auc = compute_per_writer_auc(diagnostic_scores)

    return ProtocolResult(
        val_tau_star=tau_selection["tau_star"],
        val_balanced_acc_mean=tau_selection["val_balanced_acc_mean"],
        test_summary=test_summary,
        test_summary_surds_convention=test_summary_surds,
        per_writer_auc=per_writer_auc,
        num_references=num_references,
    )


In [ ]:
%%writefile /kaggle/working/supervised_src/trainer_kaggle.py
"""Downstream supervised verification trainer - Kaggle version of the local
project's training/trainer.py, specialized to Step 4 Cell B (double-margin
combined-distance loss, stage4 unfrozen) since that is the only rung this
notebook runs - the local trainer.py's other LOSS_TYPE branches
(dual_triplet / double_margin) are omitted rather than ported unused.

All hyperparameters and paths come from kaggle_config.py (the notebook's
single editable cell) - nothing here should need touching for a normal
re-run; edit kaggle_config.py instead.
"""

from __future__ import annotations

import functools

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from writer_splits import DATA_ROOT, get_writer_split
from dual_triplet_dataset import DualTripletDataset
from fixed_dual_triplet_dataset import FixedDualTripletDataset, build_fixed_triplet_records
from pair_dataset import PairDataset
from embedding_model import load_downstream_model
from double_margin_distance_loss import DoubleMarginDistanceLoss
from train_validation import train_and_validate_model
from run_one_epoch_combined import run_one_epoch_combined
from protocol import build_verification_callback, make_combined_batch_distance_fn, make_combined_distance_fn

import kaggle_config as cfg


def _get_optimizer_param_groups(
    model: nn.Module, head_lr: float, encoder_lr: float, weight_decay: float,
) -> list[dict]:
    """Splits trainable parameters into up to 4 groups: {encoder, head} x
    {weight-decayed, not}. The encoder gets `encoder_lr` (much lower - see
    kaggle_config.ENCODER_LEARNING_RATE), the projector/local_projection
    get `head_lr`."""
    section_lr = {"encoder": encoder_lr, "head": head_lr}
    grouped_params: dict[tuple[str, bool], list] = {
        (section, is_decay): [] for section in section_lr for is_decay in (True, False)
    }
    for param_name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        section = "encoder" if param_name.startswith("encoder.") else "head"
        is_decay = not (param.ndim == 1 or param_name.endswith(".bias"))
        grouped_params[(section, is_decay)].append(param)

    return [
        {"params": params, "lr": section_lr[section], "weight_decay": weight_decay if is_decay else 0.0}
        for (section, is_decay), params in grouped_params.items()
        if params
    ]


def train() -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type != "cuda":
        print("WARNING: no GPU detected - check the notebook's Accelerator setting (should be GPU T4 x2).")

    split = get_writer_split(cfg.DATASET_NAME, fold=cfg.FOLD)
    print(
        f"Writers - train: {len(split.train_writer_ids)} | "
        f"val: {len(split.validation_writer_ids)} | "
        f"test: {len(split.test_writer_ids)}"
    )

    dataset_dir = DATA_ROOT / cfg.DATASET_NAME

    train_dataset = DualTripletDataset(
        split.train_writer_ids, dataset_dir, cfg.DATASET_NAME, seed=cfg.TRAIN_SEED,
    )
    val_dataset = FixedDualTripletDataset(
        build_fixed_triplet_records(
            split.validation_writer_ids, dataset_dir, cfg.DATASET_NAME,
            tuples_per_anchor=cfg.VAL_TUPLES_PER_ANCHOR, seed=cfg.VAL_SEED,
        )
    )
    # double_margin_combined: decompose each quadruple into its 3 labeled pairs.
    train_dataset = PairDataset(train_dataset)
    val_dataset = PairDataset(val_dataset)
    print(f"Dataset sizes - train: {len(train_dataset)} | val: {len(val_dataset)}")

    train_loader = DataLoader(
        train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=False, num_workers=cfg.NUM_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, drop_last=False, num_workers=cfg.NUM_WORKERS,
    )
    print(f"Batches per epoch - train: {len(train_loader)} | val: {len(val_loader)}")

    model = load_downstream_model(
        cfg.RUN_NAME, cfg.CHECKPOINT_EPOCH, device,
        trainable_encoder_stages=cfg.TRAINABLE_ENCODER_STAGES,
        projector_hidden_dim=cfg.PROJECTOR_HIDDEN_DIM,
        embedding_dim=cfg.EMBEDDING_DIM,
        norm_type=cfg.NORM_TYPE,
        local_embedding_dim=cfg.LOCAL_EMBEDDING_DIM,
    )

    optimizer = torch.optim.AdamW(
        _get_optimizer_param_groups(
            model, head_lr=cfg.LEARNING_RATE, encoder_lr=cfg.ENCODER_LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY,
        ),
    )

    print(f"Run tag: {cfg.RUN_TAG}")
    print(f"Results:  {cfg.RESULTS_DIR}")
    print(f"Batch size: {cfg.BATCH_SIZE} | Head LR: {cfg.LEARNING_RATE} | Encoder LR: {cfg.ENCODER_LEARNING_RATE}")
    print(f"Epochs: {cfg.NUM_EPOCHS} (patience {cfg.PATIENCE}, min {cfg.MIN_EPOCHS})")
    print(
        f"Margins (combined): m={cfg.MARGIN_M_COMBINED}, n={cfg.MARGIN_N_COMBINED} "
        f"| lambda_0={cfg.LAMBDA_0} | local_dim={cfg.LOCAL_EMBEDDING_DIM}"
    )

    loss_function = DoubleMarginDistanceLoss(margin_m=cfg.MARGIN_M_COMBINED, margin_n=cfg.MARGIN_N_COMBINED)
    run_epoch_fn = functools.partial(
        run_one_epoch_combined, lambda_0=cfg.LAMBDA_0,
        sinkhorn_epsilon=cfg.SINKHORN_EPSILON, sinkhorn_iterations=cfg.SINKHORN_ITERATIONS,
    )
    eval_distance_fn = make_combined_distance_fn(
        model, lambda_0=cfg.LAMBDA_0, sinkhorn_epsilon=cfg.SINKHORN_EPSILON, sinkhorn_iterations=cfg.SINKHORN_ITERATIONS,
    )
    eval_batch_distance_fn = make_combined_batch_distance_fn(
        model, lambda_0=cfg.LAMBDA_0, sinkhorn_epsilon=cfg.SINKHORN_EPSILON, sinkhorn_iterations=cfg.SINKHORN_ITERATIONS,
    )

    verification_callback = build_verification_callback(
        dataset_dir, cfg.DATASET_NAME, split.validation_writer_ids,
        eval_distance_fn,
        num_references=cfg.VERIFICATION_NUM_REFERENCES,
        seeds=cfg.VERIFICATION_SEEDS,
        batch_distance_fn=eval_batch_distance_fn,
        batch_size=cfg.VERIFICATION_BATCH_SIZE,
    )
    print(
        f"Checkpoint selection: K-reference validation AUC "
        f"(K={cfg.VERIFICATION_NUM_REFERENCES}, {len(cfg.VERIFICATION_SEEDS)} draws/epoch) "
        f"[batched, batch_size={cfg.VERIFICATION_BATCH_SIZE}]"
    )

    history_csv_path = cfg.RESULTS_DIR / "training_history.csv"
    model_dir = cfg.RESULTS_DIR / "checkpoints"

    history_df = train_and_validate_model(
        model=model,
        train_dataset=train_dataset,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_function=loss_function,
        optimizer=optimizer,
        epochs=cfg.NUM_EPOCHS,
        device=device,
        history_csv_path=history_csv_path,
        model_dir=model_dir,
        periodic_save_frequency=cfg.PERIODIC_SAVE_FREQUENCY,
        verification_callback=verification_callback,
        run_epoch_fn=run_epoch_fn,
        patience=cfg.PATIENCE,
        min_epochs=cfg.MIN_EPOCHS,
    )

    print("Training complete.")
    print(history_df)


## 5. Run training

In [ ]:
from trainer_kaggle import train

train()


## 6. Verify results

In [ ]:
import pandas as pd

import kaggle_config as cfg

history = pd.read_csv(cfg.RESULTS_DIR / "training_history.csv")
print(history.to_string(index=False))

checkpoint_dir = cfg.RESULTS_DIR / "checkpoints"
print("\nCheckpoints saved:")
for path in sorted(checkpoint_dir.glob("*.pt")):
    print(f"  {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

best_epoch = history.loc[history["val_proto_auc_mean"].idxmax()]
print(f"\nBest epoch by val_proto_auc_mean: {int(best_epoch['epoch'])} (AUC={best_epoch['val_proto_auc_mean']:.4f})")
